# Review Morning Desk: full pipeline
Runtime > Run all (Ctrl+F9). About 6-9 minutes on the free CPU runtime. The last cell downloads `reviews.db` and `dashboard_data.json`.

In [ ]:
!pip install -q sentence-transformers

## Step 1: mock Google Play review stream

In [ ]:
%%writefile generate_play_reviews.py
"""
Step 1 - Fake Google Play review stream
=======================================

Simulates what our collector would receive from the Google Play Developer
API (reviews.list) if it polled every app every day for 60 days.

Why this shape:
  * The real API only returns reviews created or modified in the LAST 7 DAYS,
    so a real system must poll daily and keep its own history. We simulate the
    daily pulls, including reviews that get EDITED later (same reviewId,
    newer lastModified, new rating/text).
  * The API gives NO country, only reviewerLanguage. It does give app version
    and rich device metadata, which is what lets us pin a bug to a release
    and to specific phones.
  * Ground truth (which scenario we planted) is written to a SEPARATE file,
    so the automation physically can't see it. It is only used at the end
    to measure what the system caught.

Run:  python generate_play_reviews.py      (or paste into one Colab cell)

Outputs (folder ./data):
  apps.csv                 our internal app catalog (genre, audience)
  play_api_pulls.jsonl     one line per review object returned by the API,
                           per daily pull, in the API's JSON shape
  reviews_latest.csv       flat, human-readable view (latest version of each
                           review) - for opening in Sheets / Excel
  ground_truth.csv         review_id -> planted scenario (evaluation only)
"""

import csv
import json
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

SEED = 7
START = datetime(2026, 8, 1, tzinfo=timezone.utc)
DAYS = 60
NOISE_RATE = 0.15
CHILD_SAFETY_COUNT = 20
OUT = "data"

rng = random.Random(SEED)

# ---------------------------------------------------------------------------
# App catalog (internal metadata - not from the store)
# ---------------------------------------------------------------------------
APPS = [
    # package,                       name,            genre,         family,       audience, reviews/day
    ("com.example.mergemeadow",     "Merge Meadow",  "casual_merge", "game",       "general", 110),
    ("com.example.smashrunner",     "Smash Runner",  "hypercasual",  "game",       "general", 90),
    ("com.example.tiletwist",       "Tile Twist",    "casual_puzzle","game",       "general", 25),
    ("com.example.pelegame",        "PeleGame",      "meta_world",   "meta_world", "kids_teens", 60),
]
APP = {a[0]: dict(zip(["package", "name", "genre", "family", "audience", "daily"], a)) for a in APPS}

# version timeline: (first day live, versionName, versionCode)
VERSIONS = {
    "com.example.mergemeadow": [(0, "2.3.0", 2300), (30, "2.4.0", 2400), (38, "2.4.1", 2410)],
    "com.example.smashrunner": [(0, "1.8.2", 1082), (20, "1.9.0", 1090)],
    "com.example.tiletwist":   [(0, "3.0.4", 3004), (45, "3.1.0", 3100)],
    "com.example.pelegame":    [(0, "5.1.0", 5100), (35, "5.2.0", 5200)],
}

# reviewerLanguage share (the API gives language, not country)
LANGS = [("en", 0.45), ("pt", 0.22), ("de", 0.18), ("he", 0.15)]

# device -> deviceMetadata (fake but realistic structure)
DEVICES = {
    "a14":      {"productName": "Galaxy A14 (a14)", "manufacturer": "Samsung", "ramMb": 4096, "cpuMake": "Mediatek"},
    "a54x":     {"productName": "Galaxy A54 5G (a54x)", "manufacturer": "Samsung", "ramMb": 6144, "cpuMake": "Samsung"},
    "e3q":      {"productName": "Galaxy S24 Ultra (e3q)", "manufacturer": "Samsung", "ramMb": 12288, "cpuMake": "Qualcomm"},
    "shiba":    {"productName": "Pixel 8 (shiba)", "manufacturer": "Google", "ramMb": 8192, "cpuMake": "Google"},
    "sapphire": {"productName": "Redmi Note 13 (sapphire)", "manufacturer": "Xiaomi", "ramMb": 6144, "cpuMake": "Qualcomm"},
    "cancunf":  {"productName": "moto g54 5G (cancunf)", "manufacturer": "Motorola", "ramMb": 8192, "cpuMake": "Mediatek"},
}
DEVICE_WEIGHTS = {"a14": 22, "a54x": 18, "e3q": 12, "shiba": 10, "sapphire": 20, "cancunf": 18}
BUGGY_DEVICES = {"a14", "a54x", "sapphire"}          # Merge Meadow 2.4.0 crash
OS_LEVELS = [(33, 15), (34, 40), (35, 35), (36, 10)]   # Android API level, weight

AUTHORS = ["Daniel", "Maya", "Lucas", "Anna", "Noa", "Pedro", "Lena", "Itay", "Sofia", "Max",
           "Yael", "Gabriel", "Mia", "Jonas", "Tamar", "Rafael", "Emma", "Omer", "Julia", "Ben"]

# ---------------------------------------------------------------------------
# Text bank:  scenario -> family ("game" / "meta_world" / "any") -> lang -> templates
# Placeholders: {app} {version} {level} {amount} {age}
# ---------------------------------------------------------------------------
BANK = {
    "background_positive": {
        "game": {
            "en": ["Love this game, perfect for short breaks.", "So relaxing, I play {app} every night before bed.",
                   "Great levels and cute graphics.", "Addictive in the best way. Already at level {level}.",
                   "Simple, fun and smooth. Exactly what I needed.", "One of the best casual games on my phone.",
                   "Fun way to pass time on the bus.", "My wife and I compete on who gets further."],
            "de": ["Tolles Spiel, perfekt für kurze Pausen.", "Sehr entspannend, ich spiele {app} jeden Abend.",
                   "Schöne Level und süße Grafik.", "Macht süchtig, bin schon bei Level {level}.",
                   "Einfach, lustig und läuft flüssig.", "Eines der besten Spiele auf meinem Handy."],
            "pt": ["Amo esse jogo, perfeito para passar o tempo.", "Muito relaxante, jogo {app} toda noite.",
                   "Fases legais e gráficos lindos.", "Viciante demais, já estou no nível {level}.",
                   "Simples, divertido e roda liso.", "Um dos melhores jogos do meu celular."],
            "he": ["משחק מעולה, מושלם להפסקות קצרות.", "ממש מרגיע, אני משחק ב־{app} כל ערב.",
                   "שלבים יפים וגרפיקה חמודה.", "ממכר ברמות, כבר הגעתי לשלב {level}.",
                   "פשוט, כיף ורץ חלק.", "אחד המשחקים הכי טובים בטלפון שלי."],
        },
        "meta_world": {
            "en": ["So many games in one place, my kids love it.", "Great way to hang out with friends online.",
                   "Building my own world is super fun.", "Lots of cool avatars and events.",
                   "New games every week, never boring."],
            "de": ["So viele Spiele an einem Ort, meine Kinder lieben es.", "Super, um online mit Freunden abzuhängen.",
                   "Eine eigene Welt zu bauen macht richtig Spaß.", "Viele coole Avatare und Events."],
            "pt": ["Muitos jogos num lugar só, meus filhos amam.", "Ótimo para ficar com os amigos online.",
                   "Construir meu próprio mundo é muito divertido.", "Muitos avatares e eventos legais."],
            "he": ["המון משחקים במקום אחד, הילדים שלי מתים על זה.", "דרך מעולה לבלות עם חברים אונליין.",
                   "לבנות עולם משלי זה ממש כיף.", "הרבה דמויות ואירועים מגניבים."],
        },
    },
    "background_complaint": {
        "game": {
            "en": ["Levels get repetitive after a while.", "Too hard after level {level}, I'm stuck.",
                   "Lives refill way too slowly.", "Fun but the boosters are too expensive.",
                   "Needs more content, I finished everything already.", "Battery drains fast when playing."],
            "de": ["Die Level wiederholen sich irgendwann.", "Ab Level {level} viel zu schwer.",
                   "Leben laden viel zu langsam auf.", "Die Booster sind zu teuer.", "Akku ist schnell leer."],
            "pt": ["As fases ficam repetitivas.", "Muito difícil depois do nível {level}.",
                   "As vidas demoram demais para recarregar.", "Os boosters são caros demais.", "A bateria acaba rápido."],
            "he": ["השלבים חוזרים על עצמם אחרי זמן מה.", "קשה מדי אחרי שלב {level}, אני תקוע.",
                   "החיים מתמלאים לאט מדי.", "הבוסטרים יקרים מדי.", "הסוללה נגמרת מהר כשמשחקים."],
        },
        "meta_world": {
            "en": ["Servers lag when lots of players join.", "Loading between games takes forever.",
                   "Hard to find my friends in the lobby.", "Too many items are paid only."],
            "de": ["Server laggen, wenn viele Spieler da sind.", "Laden zwischen den Spielen dauert ewig.",
                   "Zu viele Items kosten Geld."],
            "pt": ["Os servidores travam com muitos jogadores.", "O carregamento entre os jogos demora muito.",
                   "Muitos itens são só pagos."],
            "he": ["השרתים נתקעים כשהרבה שחקנים נכנסים.", "הטעינה בין המשחקים לוקחת נצח.",
                   "יותר מדי פריטים רק בתשלום."],
        },
    },
    "version_crash": {"any": {
        "en": ["Since the last update the game crashes every time I open it.",
               "Crashes on the loading screen after updating to {version}.",
               "Updated to {version} and now it closes instantly.",
               "Can't play at all since the update, black screen then crash.",
               "Game keeps freezing and closing since the new version."],
        "de": ["Seit dem letzten Update stürzt das Spiel beim Öffnen ab.",
               "Nach dem Update auf {version} hängt es im Ladebildschirm.",
               "Seit {version} geht die App sofort wieder zu.", "Startet nach dem Update gar nicht mehr."],
        "pt": ["Depois da última atualização o jogo fecha toda vez que abro.",
               "Trava na tela de carregamento depois de atualizar para {version}.",
               "Atualizei para {version} e agora fecha na hora.", "Depois da atualização nem abre mais."],
        "he": ["מאז העדכון האחרון המשחק קורס בכל פעם שאני פותח אותו.", "נתקע במסך הטעינה אחרי העדכון ל־{version}.",
               "עדכנתי ל־{version} ועכשיו הוא נסגר מיד.", "אחרי העדכון הוא בכלל לא נפתח."],
    }},
    "progress_lost": {"any": {
        "en": ["After the update all my progress is gone, back to level 1!", "Lost 200 levels after updating. Where is my save?",
               "Update {version} wiped my progress. Please restore it."],
        "de": ["Nach dem Update ist mein ganzer Fortschritt weg!", "Update {version} hat meinen Spielstand gelöscht."],
        "pt": ["Depois da atualização perdi todo o meu progresso!", "A versão {version} apagou meu progresso."],
        "he": ["אחרי העדכון כל ההתקדמות שלי נמחקה, חזרתי לשלב 1!", "העדכון ל־{version} מחק לי את כל השלבים."],
    }},
    "child_safety": {"any": {
        "en": ["An adult player kept sending private messages to my {age} year old and asked him to talk on another app. Please moderate the chat!",
               "Someone in the chat asked my daughter ({age}) personal questions and where she lives. Very worried.",
               "A stranger offered my son free coins if he added him on a different messaging app. This is not safe for kids.",
               "I reported a user who was messaging young kids inappropriately and nothing happened. Parents, be careful."],
        "de": ["Ein erwachsener Spieler hat meinem {age}-jährigen Sohn private Nachrichten geschickt und wollte in einer anderen App weiterschreiben.",
               "Jemand im Chat hat meine Tochter ({age}) gefragt, wo sie wohnt. Ich bin sehr besorgt."],
        "pt": ["Um jogador adulto mandou mensagens privadas para meu filho de {age} anos e pediu para conversar em outro app.",
               "Alguém no chat perguntou para minha filha ({age}) onde ela mora. Estou muito preocupada."],
        "he": ["שחקן מבוגר שלח לבן שלי בן ה־{age} הודעות פרטיות וביקש לעבור לדבר באפליקציה אחרת. תפקחו על הצ'אט!",
               "מישהו בצ'אט שאל את הבת שלי ({age}) איפה היא גרה. אני מאוד מודאגת.",
               "זר הציע לבן שלי מטבעות בחינם אם יוסיף אותו באפליקציה אחרת. זה לא בטוח לילדים."],
    }},
    "iap_not_received": {"any": {
        "en": ["Bought {amount} worth of coins and never received them.", "Paid for the gem pack, money was taken but nothing in my account.",
               "Purchase went through but no coins. Support doesn't answer.", "Charged twice for the same pack and got nothing."],
        "de": ["Habe Münzen für {amount} gekauft und nie bekommen.", "Bezahlt, Geld abgebucht, aber nichts im Konto."],
        "pt": ["Comprei {amount} em moedas e nunca recebi.", "Paguei o pacote, o dinheiro saiu mas não apareceu nada."],
        "he": ["קניתי מטבעות ב־{amount} ולא קיבלתי אותם.", "שילמתי על החבילה, הכסף ירד ושום דבר לא הגיע לחשבון."],
    }},
    "ads_chronic": {"any": {
        "en": ["An ad after every single run, it's unbearable.", "Way too many ads, I watch more ads than I play.",
               "30 second ads you can't skip. Uninstalling.", "Paid to remove ads and they still show up."],
        "de": ["Nach jedem Lauf Werbung, unerträglich.", "Viel zu viel Werbung, mehr Werbung als Spiel."],
        "pt": ["Propaganda depois de cada corrida, insuportável.", "Anúncio demais, vejo mais anúncio do que jogo."],
        "he": ["פרסומת אחרי כל ריצה, בלתי נסבל.", "יותר מדי פרסומות, אני רואה יותר פרסומות משאני משחק."],
    }},
    "feature_avatars": {"any": {
        "en": ["Please add more avatar customization, like hairstyles and outfits.", "Would love to design my own avatar clothes.",
               "More avatar options please! Everyone looks the same.", "Add pets for avatars, that would be amazing."],
        "de": ["Bitte mehr Möglichkeiten, den Avatar anzupassen.", "Mehr Avatar-Optionen bitte! Alle sehen gleich aus."],
        "pt": ["Por favor, mais opções para personalizar o avatar.", "Mais opções de avatar! Todo mundo parece igual."],
        "he": ["תוסיפו בבקשה עוד אפשרויות לעצב את הדמות.", "עוד אפשרויות לדמות! כולם נראים אותו דבר."],
    }},
    "high_value": {"any": {
        "en": ["I've spent over {amount} on this game and the latest changes ruined it.",
               "Playing for 3 years, level {level}, spent a lot of money. Very disappointed lately.",
               "VIP player here. If this isn't fixed I'm leaving."],
        "de": ["Ich habe über {amount} in dieses Spiel gesteckt und die letzten Änderungen haben alles kaputt gemacht.",
               "VIP-Spieler hier. Wenn das nicht behoben wird, höre ich auf."],
        "pt": ["Já gastei mais de {amount} nesse jogo e as últimas mudanças estragaram tudo.", "Sou jogador VIP. Se não arrumarem, vou sair."],
        "he": ["הוצאתי על המשחק הזה יותר מ־{amount} והשינויים האחרונים הרסו אותו.", "שחקן VIP כאן. אם זה לא יתוקן אני עוזב."],
    }},
    "rating_mismatch": {"any": {
        "en": ["Love it but it keeps crashing every few minutes, please fix.", "Great game, but it freezes and closes all the time."],
        "de": ["Liebe es, aber es stürzt alle paar Minuten ab."],
        "pt": ["Amo, mas fecha sozinho toda hora."],
        "he": ["אוהב את המשחק אבל הוא קורס כל כמה דקות, תתקנו."],
    }},
    # --- URGENT: things that can hurt the app's standing in the store (planted at the end) ---
    "policy_ads": {"any": {
        "en": ["My 9 year old saw a casino gambling ad in this game. Ads like that in a kids game are not ok.",
               "An adult content ad popped up while my son was playing. Reporting this to Google.",
               "The ad installed another app on my phone without asking. Isn't that against Google's rules?",
               "Ads have a fake close button that opens the store. Feels like a scam."],
        "de": ["Mein Kind hat in diesem Spiel Werbung für Glücksspiel gesehen. Das geht gar nicht.",
               "Die Werbung hat ohne zu fragen eine andere App installiert. Ich melde das bei Google."],
        "pt": ["Apareceu um anúncio de cassino de apostas para o meu filho de 8 anos. Inaceitável.",
               "O anúncio instalou outro app no meu celular sem pedir. Vou denunciar para o Google."],
        "he": ["הבן שלי בן 9 ראה במשחק פרסומת להימורים בקזינו. זה לא בסדר במשחק לילדים.",
               "קפצה פרסומת עם תוכן למבוגרים בזמן שהילד שיחק. אני מדווח לגוגל."],
    }},
    "legal_threat": {"any": {
        "en": ["You charged my card three times and ignore my emails. My lawyer will be in touch.",
               "This is fraud. I'm filing a complaint with consumer protection."],
        "de": ["Dreimal abgebucht und keine Antwort. Ich schalte meinen Anwalt ein."],
        "pt": ["Isso é golpe. Vou abrir uma reclamação no Procon e procurar um advogado."],
        "he": ["חייבתם אותי שלוש פעמים ולא עונים. אני פונה לעורך דין ולרשות להגנת הצרכן."],
    }},
    "privacy_risk": {"any": {
        "en": ["Why does a kids game ask for my child's location and contacts? Deleting it.",
               "The app keeps tracking my kid's location even when closed. What do you do with this data?"],
        "de": ["Warum will ein Kinderspiel den Standort und die Kontakte meines Kindes? Datenschutz?"],
        "pt": ["Por que um jogo infantil pede a localização e os contatos do meu filho? E a privacidade?"],
        "he": ["למה משחק לילדים מבקש גישה למיקום ולאנשי הקשר של הילד שלי? מה אתם עושים עם המידע?"],
    }},
    "noise_off_topic": {"any": {
        "en": ["Who won the match yesterday?", "Wrong app, I meant to rate the bus app."],
        "de": ["Falsche App, wollte die Wetter-App bewerten."],
        "pt": ["Alguém sabe uma receita de bolo?"],
        "he": ["טעיתי באפליקציה, רציתי לדרג את אפליקציית מזג האוויר."],
    }},
    "noise_spam": {"any": {
        "en": ["FREE COINS 100% working!!! visit free-coins-gen.example", "Get unlimited gems now at gemhack.example"],
        "de": ["GRATIS MÜNZEN!!! jetzt auf free-coins-gen.example"],
        "pt": ["MOEDAS GRÁTIS!!! acesse free-coins-gen.example"],
        "he": ["מטבעות בחינם!!! היכנסו ל־free-coins-gen.example"],
    }},
}
NOISE_SHORT = {"en": ["ok", "nice", "good", "meh", "cool"], "de": ["gut", "top", "ok"],
               "pt": ["bom", "legal", "ok"], "he": ["אחלה", "סבבה", "בסדר"]}
NOISE_EMOJI = ["👍", "🔥🔥", "😍", "👎", "💯", "🙂"]
NOISE_GIBBERISH = ["asdfgh", "jjjjjjj", "qwerty", "lol lol lol", "....", "aaaaaaa"]

# small openers/closers so repeated templates don't look identical
OPENERS = {"en": ["", "", "", "Honestly, ", "Ugh. ", "Hi devs, "], "de": ["", "", "", "Ehrlich gesagt: ", "Hallo, "],
           "pt": ["", "", "", "Sinceramente, ", "Oi, "], "he": ["", "", "", "האמת, ", "היי, "]}
CLOSERS = {"neg": {"en": ["", "", " Please fix.", " Very frustrating."], "de": ["", "", " Bitte beheben."],
                   "pt": ["", "", " Por favor arrumem."], "he": ["", "", " תתקנו בבקשה."]},
           "pos": {"en": ["", "", " Keep it up!", " Thanks devs!"], "de": ["", "", " Weiter so!"],
                   "pt": ["", "", " Continuem assim!"], "he": ["", "", " תמשיכו ככה!"]}}

AMOUNTS = {"en": ["$4.99", "$19.99", "$49.99"], "de": ["4,99 €", "19,99 €"], "pt": ["R$ 24,90", "R$ 99,90"], "he": ["₪19.90", "₪79.90"]}
BIG_AMOUNTS = {"en": ["$300", "$800", "$1,500"], "de": ["300 €", "800 €"], "pt": ["R$ 1.500", "R$ 4.000"], "he": ["₪1,000", "₪3,000"]}

RATINGS = {"background_positive": (4, 5), "background_complaint": (2, 3), "version_crash": (1, 2),
           "progress_lost": (1, 1), "child_safety": (1, 1), "iap_not_received": (1, 2), "ads_chronic": (1, 3),
           "feature_avatars": (3, 5), "high_value": (1, 2), "rating_mismatch": (5, 5),
           "policy_ads": (1, 1), "legal_threat": (1, 1), "privacy_risk": (1, 2),
           "noise_off_topic": (1, 5), "noise_spam": (5, 5), "noise_gibberish": (1, 5)}


# ---------------------------------------------------------------------------
def wpick(pairs):
    return rng.choices([p[0] for p in pairs], weights=[p[1] for p in pairs])[0]


def version_for(pkg, day):
    live = [(v, c) for d, v, c in VERSIONS[pkg] if d <= day]
    if len(live) > 1 and rng.random() < 0.15:      # some users haven't updated yet
        return live[-2]
    return live[-1]


def choose_scenario(pkg, day, device, version):
    if rng.random() < NOISE_RATE:
        return rng.choices(["noise_short", "noise_emoji", "noise_gibberish", "noise_off_topic", "noise_spam"],
                           weights=[40, 20, 10, 15, 15])[0]
    if pkg == "com.example.mergemeadow":
        if version == "2.4.0" and rng.random() < (0.60 if device in BUGGY_DEVICES else 0.04):
            return "version_crash"
        if version == "2.4.1" and device in BUGGY_DEVICES and rng.random() < 0.04:
            return "version_crash"
    if pkg == "com.example.tiletwist" and version == "3.1.0" and rng.random() < 0.35:
        return "progress_lost"
    if pkg == "com.example.pelegame":
        if day >= 40 and rng.random() < 0.12:
            return "iap_not_received"
        if rng.random() < 0.02 + 0.13 * day / (DAYS - 1):
            return "feature_avatars"
    if pkg == "com.example.smashrunner" and rng.random() < 0.18:
        return "ads_chronic"
    if rng.random() < 0.006:
        return "rating_mismatch"
    return "background_positive" if rng.random() < 0.65 else "background_complaint"


def write_text(scenario, family, lang, app_name, version, rating):
    if scenario == "noise_short":
        return rng.choice(NOISE_SHORT[lang])
    if scenario == "noise_emoji":
        return rng.choice(NOISE_EMOJI)
    if scenario == "noise_gibberish":
        return rng.choice(NOISE_GIBBERISH)
    pool = BANK[scenario].get(family) or BANK[scenario]["any"]
    tpl = rng.choice(pool.get(lang) or pool["en"])
    text = tpl.format(app=app_name, version=version, level=rng.randint(20, 900), age=rng.randint(8, 12),
                      amount=rng.choice((BIG_AMOUNTS if scenario == "high_value" else AMOUNTS)[lang]))
    if scenario.startswith("noise") or scenario in ("child_safety", "policy_ads", "legal_threat", "privacy_risk"):
        return text
    tone = "neg" if rating <= 2 else "pos" if rating >= 4 else None
    opener = rng.choice(OPENERS[lang])
    if opener and lang in ("en", "pt") and not text.startswith("I"):
        text = text[0].lower() + text[1:]
    text = opener + text + (rng.choice(CLOSERS[tone][lang]) if tone else "")
    if lang == "en" and rng.random() < 0.05:
        text = text.lower()
    return text


def rating_for(scenario, text):
    if scenario == "noise_short":
        return rng.randint(3, 5)
    if scenario == "noise_emoji":
        return 1 if text == "👎" else rng.randint(4, 5)
    lo, hi = RATINGS[scenario]
    return rng.randint(lo, hi)


def thumbs(scenario):
    if scenario in ("child_safety", "policy_ads", "privacy_risk", "legal_threat"):
        return rng.randint(5, 60)
    if scenario in ("version_crash", "iap_not_received", "progress_lost"):
        return int(rng.expovariate(1 / 6))
    return int(rng.expovariate(1 / 1.5))


def ts(day):
    return START + timedelta(days=day, hours=rng.randint(0, 23), minutes=rng.randint(0, 59), seconds=rng.randint(0, 59))


def api_object(review_id, author, c):
    """Exactly the nesting the Play Developer API returns for reviews.list."""
    dm = DEVICES[c["device"]]
    return {
        "reviewId": review_id,
        "authorName": author,
        "comments": [{"userComment": {
            "text": "\t" + c["text"],                      # the API prefixes text with a tab
            "lastModified": {"seconds": str(int(c["ts"].timestamp())), "nanos": 0},
            "starRating": c["rating"],
            "reviewerLanguage": c["lang"],
            "device": c["device"],
            "androidOsVersion": c["os"],
            "appVersionCode": c["vcode"],
            "appVersionName": c["vname"],
            "thumbsUpCount": c["thumbs"],
            "thumbsDownCount": 0,
            "deviceMetadata": {"productName": dm["productName"], "manufacturer": dm["manufacturer"],
                               "deviceClass": "phone", "ramMb": dm["ramMb"], "cpuMake": dm["cpuMake"]},
        }}],
    }


# ---------------------------------------------------------------------------
def generate():
    reviews = []          # each: dict with versions = [comment, (edited comment)]
    next_id = Counter()

    def new_review(pkg, day, scenario=None):
        app = APP[pkg]
        lang = wpick(LANGS)
        device = rng.choices(list(DEVICE_WEIGHTS), weights=list(DEVICE_WEIGHTS.values()))[0]
        vname, vcode = version_for(pkg, day)
        scenario = scenario or choose_scenario(pkg, day, device, vname)
        text0 = write_text(scenario, app["family"], lang, app["name"], vname, 3)  # tone fixed after rating
        rating = rating_for(scenario, text0)
        text = text0 if scenario.startswith("noise") else write_text(scenario, app["family"], lang, app["name"], vname, rating)
        next_id[pkg] += 1
        rid = f"gp:{pkg.split('.')[-1]}:{next_id[pkg]:06d}"
        return {"id": rid, "pkg": pkg, "author": rng.choice(AUTHORS), "scenario": scenario,
                "versions": [{"text": text, "rating": rating, "lang": lang, "device": device,
                              "os": wpick(OS_LEVELS), "vname": vname, "vcode": vcode,
                              "thumbs": thumbs(scenario), "ts": ts(day)}]}

    for day in range(DAYS):
        for pkg, app in APP.items():
            n = rng.randint(int(app["daily"] * 0.85), int(app["daily"] * 1.15))
            if pkg == "com.example.mergemeadow" and 30 <= day <= 37:
                n = int(n * 1.5)        # bad release -> more people write
            if pkg == "com.example.tiletwist" and 45 <= day <= 52:
                n = int(n * 1.4)
            for _ in range(n):
                reviews.append(new_review(pkg, day))

    # child safety: rare, spread over the whole period
    for day in sorted(rng.sample(range(3, DAYS), CHILD_SAFETY_COUNT)):
        reviews.append(new_review("com.example.pelegame", day, "child_safety"))

    # EDITS: happy players who come back and downgrade.
    #   (a) Merge Meadow 5-star reviews from before 2.4.0, edited to crash reports during 2.4.0
    #   (b) long-time / paying players edited 5 -> 1 (high_value)
    pre = [r for r in reviews if r["pkg"] == "com.example.mergemeadow" and r["scenario"] == "background_positive"
           and r["versions"][0]["ts"] < START + timedelta(days=28) and r["versions"][0]["device"] in BUGGY_DEVICES]
    for r in rng.sample(pre, 25):
        v0 = r["versions"][0]
        day = rng.randint(30, 37)
        text = write_text("version_crash", "game", v0["lang"], "Merge Meadow", "2.4.0", 1)
        r["versions"].append({**v0, "text": text, "rating": 1, "vname": "2.4.0", "vcode": 2400,
                              "thumbs": thumbs("version_crash"), "ts": ts(day)})
        r["scenario"] = "version_crash"
    pool = [r for r in reviews if r["scenario"] == "background_positive" and r["versions"][0]["ts"] < START + timedelta(days=40)]
    for r in rng.sample(pool, 30):
        v0 = r["versions"][0]
        day = rng.randint(max(41, (v0["ts"] - START).days + 5), DAYS - 1)
        app = APP[r["pkg"]]
        vname, vcode = VERSIONS[r["pkg"]][-1][1:]
        text = write_text("high_value", app["family"], v0["lang"], app["name"], vname, 1)
        r["versions"].append({**v0, "text": text, "rating": 1, "vname": vname, "vcode": vcode,
                              "thumbs": thumbs("high_value"), "ts": ts(day)})
        r["scenario"] = "high_value"

    # URGENT store-risk reports, added last so everything above stays identical:
    #   Smash Runner: a bad ad network starts showing gambling / adult ads and auto-installs (days 50-57)
    #   PeleGame: privacy questions about kids' location/contacts, and legal threats over charges
    for day in sorted(rng.choices(range(50, 58), k=14)):
        reviews.append(new_review("com.example.smashrunner", day, "policy_ads"))
    for day in sorted(rng.sample(range(25, 55), 5)):
        reviews.append(new_review("com.example.pelegame", day, "privacy_risk"))
    for day in sorted(rng.sample(range(43, 59), 6)):
        reviews.append(new_review("com.example.pelegame", day, "legal_threat"))
    return reviews


def daily_pulls(reviews):
    """What a daily poller sees: every review created/modified in the last 7 days.
    We keep one line per (pull day, review state) - but only emit a line the first
    time each state is seen, which is what an idempotent collector would store."""
    lines = []
    for r in reviews:
        for v in r["versions"]:
            fetched = (v["ts"] + timedelta(days=1)).replace(hour=6, minute=0, second=0)
            lines.append({"_packageName": r["pkg"], "_fetchedAt": fetched.isoformat(),
                          **api_object(r["id"], r["author"], v)})
    lines.sort(key=lambda x: (x["_fetchedAt"], x["comments"][0]["userComment"]["lastModified"]["seconds"]))
    return lines


def save(reviews, lines):
    os.makedirs(OUT, exist_ok=True)
    with open(f"{OUT}/apps.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["package_name", "app_name", "genre", "genre_family", "audience"])
        for a in APPS:
            w.writerow(a[:5])
    with open(f"{OUT}/play_api_pulls.jsonl", "w", encoding="utf-8") as f:
        for line in lines:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")
    cols = ["review_id", "package_name", "app_name", "genre", "submitted_at", "last_modified", "star_rating",
            "first_star_rating", "edited", "reviewer_language", "text", "app_version_name", "app_version_code",
            "device", "device_name", "android_os_version", "thumbs_up"]
    with open(f"{OUT}/reviews_latest.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in sorted(reviews, key=lambda r: r["versions"][-1]["ts"]):
            first, last = r["versions"][0], r["versions"][-1]
            app = APP[r["pkg"]]
            w.writerow({"review_id": r["id"], "package_name": r["pkg"], "app_name": app["name"], "genre": app["genre"],
                        "submitted_at": first["ts"].isoformat(), "last_modified": last["ts"].isoformat(),
                        "star_rating": last["rating"], "first_star_rating": first["rating"],
                        "edited": len(r["versions"]) > 1, "reviewer_language": last["lang"], "text": last["text"],
                        "app_version_name": last["vname"], "app_version_code": last["vcode"], "device": last["device"],
                        "device_name": DEVICES[last["device"]]["productName"], "android_os_version": last["os"],
                        "thumbs_up": last["thumbs"]})
    with open(f"{OUT}/ground_truth.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["review_id", "true_scenario"])
        for r in reviews:
            w.writerow([r["id"], r["scenario"]])


def summary(reviews, lines):
    print(f"Reviews: {len(reviews)}   API lines (incl. edits): {len(lines)}   days: {DAYS}   "
          f"avg/day: {len(reviews) / DAYS:.0f}")
    print("By app:", dict(Counter(APP[r['pkg']]['name'] for r in reviews)))
    print("By language:", dict(Counter(r['versions'][-1]['lang'] for r in reviews)))
    print("Edited reviews:", sum(len(r["versions"]) > 1 for r in reviews))
    print("\nPlanted scenarios:")
    for s, n in Counter(r["scenario"] for r in reviews).most_common():
        print(f"  {s:22s} {n}")
    crash = [r["versions"][-1] for r in reviews if r["scenario"] == "version_crash"]
    print("\nversion_crash by version:", dict(Counter(c["vname"] for c in crash)))
    print("version_crash by device:", dict(Counter(c["device"] for c in crash).most_common()))
    tt = [r for r in reviews if r["pkg"] == "com.example.tiletwist" and r["versions"][-1]["vname"] == "3.1.0"]
    pl = sum(r["scenario"] == "progress_lost" for r in tt)
    print(f"Tile Twist 3.1.0: {pl} progress_lost of {len(tt)} reviews ({pl / len(tt):.0%})")


if __name__ == "__main__":
    revs = generate()
    pulls = daily_pulls(revs)
    save(revs, pulls)
    summary(revs, pulls)
    print(f"\nSaved to ./{OUT}/")


In [ ]:
!python generate_play_reviews.py

## Step 2: collector

In [ ]:
%%writefile collector.py
"""
Step 2 - Collector: Google Play -> SQLite, with history
=======================================================

What it does, once per day per app:
  1. Ask Google Play for "all reviews created or modified in the last 7 days"
  2. Save each review it hasn't seen
  3. If a review it HAS seen came back with a newer lastModified, the user
     edited it -> keep the old version in history and record the change
  4. Ignore exact repeats (the 7-day windows overlap, so most reviews come
     back up to 7 times - that's expected, not new data)
  5. Log every run (how many fetched / new / edited / repeats)

Why: the API forgets anything older than a week and never shows the previous
rating of an edited review. Only our own database can answer "this player was
5 stars and dropped to 1 after update 2.4.0".

In this POC, fetch_reviews() reads the fake stream from step 1. To go live you
swap that single function for fetch_reviews_live() (bottom of file).

Run:  python collector.py      -> creates reviews.db
"""

import csv
import json
import sqlite3
from collections import defaultdict
from datetime import datetime, timedelta, timezone

DB = "reviews.db"
API_DUMP = "data/play_api_pulls.jsonl"
APPS_CSV = "data/apps.csv"
WINDOW = timedelta(days=7)          # what the real API returns

SCHEMA = """
CREATE TABLE IF NOT EXISTS apps (
    package_name  TEXT PRIMARY KEY,
    app_name      TEXT,
    genre         TEXT,
    genre_family  TEXT,
    audience      TEXT
);

-- one row per review = its CURRENT state
CREATE TABLE IF NOT EXISTS reviews (
    review_id          TEXT PRIMARY KEY,
    package_name       TEXT REFERENCES apps(package_name),
    author_name        TEXT,
    text               TEXT,
    star_rating        INTEGER,
    reviewer_language  TEXT,
    app_version_name   TEXT,
    app_version_code   INTEGER,
    device             TEXT,
    device_name        TEXT,
    manufacturer       TEXT,
    ram_mb             INTEGER,
    android_os_version INTEGER,
    thumbs_up          INTEGER,
    first_seen_at      TEXT,    -- when the review was first written
    last_modified_at   TEXT,    -- latest edit
    first_star_rating  INTEGER, -- rating in the first version we saw
    times_edited       INTEGER DEFAULT 0,
    developer_reply    TEXT,    -- filled in later steps
    replied_at         TEXT
);

-- every version of every review we ever saw (the history the API doesn't keep)
CREATE TABLE IF NOT EXISTS review_versions (
    review_id         TEXT,
    last_modified_at  TEXT,
    star_rating       INTEGER,
    text              TEXT,
    app_version_name  TEXT,
    collected_at      TEXT,
    PRIMARY KEY (review_id, last_modified_at)
);

-- audit log: one row per (run, app)
CREATE TABLE IF NOT EXISTS ingestion_runs (
    run_at        TEXT,
    package_name  TEXT,
    fetched       INTEGER,
    new_reviews   INTEGER,
    edited        INTEGER,
    repeats       INTEGER
);

CREATE INDEX IF NOT EXISTS ix_reviews_app_time ON reviews(package_name, first_seen_at);
CREATE INDEX IF NOT EXISTS ix_reviews_version ON reviews(package_name, app_version_name);

-- handy view: players who came back and changed their rating
CREATE VIEW IF NOT EXISTS rating_changes AS
SELECT r.review_id, a.app_name, r.first_star_rating, r.star_rating,
       r.star_rating - r.first_star_rating AS delta,
       r.app_version_name, r.last_modified_at, r.text
FROM reviews r JOIN apps a USING (package_name)
WHERE r.times_edited > 0 AND r.star_rating <> r.first_star_rating;
"""


# ---------------------------------------------------------------------------
# Source of reviews. POC: replay the fake stream with the real API's 7-day rule.
# ---------------------------------------------------------------------------
def _load_fake_stream():
    by_pkg = defaultdict(lambda: defaultdict(list))   # pkg -> reviewId -> [versions]
    with open(API_DUMP, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            uc = obj["comments"][0]["userComment"]
            obj["_ts"] = datetime.fromtimestamp(int(uc["lastModified"]["seconds"]), timezone.utc)
            by_pkg[obj["_packageName"]][obj["reviewId"]].append(obj)
    for revs in by_pkg.values():
        for versions in revs.values():
            versions.sort(key=lambda o: o["_ts"])
    return by_pkg


_STREAM = None


def fetch_reviews(package_name, as_of):
    """Behaves like reviews.list: returns the CURRENT state of every review
    created or modified in the 7 days before `as_of`."""
    global _STREAM
    if _STREAM is None:
        _STREAM = _load_fake_stream()
    out = []
    for versions in _STREAM[package_name].values():
        current = None
        for v in versions:
            if v["_ts"] <= as_of:
                current = v
        if current and current["_ts"] > as_of - WINDOW:
            out.append(current)
    return out


# ---------------------------------------------------------------------------
# Collector
# ---------------------------------------------------------------------------
def to_row(obj):
    uc = obj["comments"][0]["userComment"]
    dm = uc.get("deviceMetadata", {})
    ts = datetime.fromtimestamp(int(uc["lastModified"]["seconds"]), timezone.utc).isoformat()
    return {
        "review_id": obj["reviewId"], "package_name": obj["_packageName"], "author_name": obj.get("authorName"),
        "text": uc["text"].strip(),               # API prefixes text with a tab
        "star_rating": uc["starRating"], "reviewer_language": uc.get("reviewerLanguage"),
        "app_version_name": uc.get("appVersionName"), "app_version_code": uc.get("appVersionCode"),
        "device": uc.get("device"), "device_name": dm.get("productName"), "manufacturer": dm.get("manufacturer"),
        "ram_mb": dm.get("ramMb"), "android_os_version": uc.get("androidOsVersion"),
        "thumbs_up": uc.get("thumbsUpCount", 0), "last_modified_at": ts,
    }


def collect(con, package_name, as_of):
    fetched = fetch_reviews(package_name, as_of)
    new = edited = repeats = 0
    now = as_of.isoformat()
    for obj in fetched:
        r = to_row(obj)
        existing = con.execute("SELECT last_modified_at FROM reviews WHERE review_id=?", (r["review_id"],)).fetchone()
        if existing is None:
            con.execute("""INSERT INTO reviews (review_id, package_name, author_name, text, star_rating,
                    reviewer_language, app_version_name, app_version_code, device, device_name, manufacturer,
                    ram_mb, android_os_version, thumbs_up, first_seen_at, last_modified_at, first_star_rating)
                    VALUES (:review_id, :package_name, :author_name, :text, :star_rating, :reviewer_language,
                    :app_version_name, :app_version_code, :device, :device_name, :manufacturer, :ram_mb,
                    :android_os_version, :thumbs_up, :last_modified_at, :last_modified_at, :star_rating)""", r)
            new += 1
        elif r["last_modified_at"] > existing[0]:
            con.execute("""UPDATE reviews SET text=:text, star_rating=:star_rating,
                    app_version_name=:app_version_name, app_version_code=:app_version_code,
                    thumbs_up=:thumbs_up, last_modified_at=:last_modified_at, times_edited=times_edited+1
                    WHERE review_id=:review_id""", r)
            edited += 1
        else:
            # same state as before; only refresh counters that change without an edit
            con.execute("UPDATE reviews SET thumbs_up=? WHERE review_id=?", (r["thumbs_up"], r["review_id"]))
            repeats += 1
            continue
        con.execute("""INSERT OR IGNORE INTO review_versions VALUES (?,?,?,?,?,?)""",
                    (r["review_id"], r["last_modified_at"], r["star_rating"], r["text"], r["app_version_name"], now))
    con.execute("INSERT INTO ingestion_runs VALUES (?,?,?,?,?,?)", (now, package_name, len(fetched), new, edited, repeats))
    return len(fetched), new, edited, repeats


def load_apps(con):
    with open(APPS_CSV, encoding="utf-8") as f:
        rows = [tuple(r.values()) for r in csv.DictReader(f)]
    con.executemany("INSERT OR REPLACE INTO apps VALUES (?,?,?,?,?)", rows)
    return [r[0] for r in rows]


def main(start=datetime(2026, 8, 2, 6, tzinfo=timezone.utc), days=60):
    con = sqlite3.connect(DB)
    con.executescript(SCHEMA)
    packages = load_apps(con)
    totals = defaultdict(int)
    for d in range(days):                       # in production: a scheduled job, once a day
        as_of = start + timedelta(days=d)
        for pkg in packages:
            f, n, e, r = collect(con, pkg, as_of)
            totals["fetched"] += f; totals["new"] += n; totals["edited"] += e; totals["repeats"] += r
        con.commit()

    print(f"Daily runs: {days} days x {len(packages)} apps")
    print(f"Fetched from API: {totals['fetched']:,}  ->  new: {totals['new']:,}  "
          f"edited: {totals['edited']}  repeats ignored: {totals['repeats']:,}")
    q = lambda sql: con.execute(sql).fetchall()
    print("\nReviews stored per app:")
    for row in q("SELECT a.app_name, COUNT(*) FROM reviews JOIN apps a USING(package_name) GROUP BY 1 ORDER BY 2 DESC"):
        print(f"  {row[0]:14s} {row[1]:,}")
    print("\nRating changes we could only see because we keep history:")
    for row in q("""SELECT app_name, first_star_rating || '->' || star_rating AS change, COUNT(*)
                    FROM rating_changes GROUP BY 1, 2 ORDER BY 3 DESC"""):
        print(f"  {row[0]:14s} {row[1]}  x{row[2]}")
    print("\nExample:")
    for row in q("SELECT app_name, first_star_rating, star_rating, app_version_name, text FROM rating_changes LIMIT 2"):
        print(f"  {row[0]}: {row[1]}★ -> {row[2]}★ on {row[3]}: \"{row[4]}\"")
    con.close()
    print(f"\nSaved {DB}")


# ---------------------------------------------------------------------------
# Going live: replace fetch_reviews with this (needs a service account with the
# "Reply to reviews" permission in Play Console).
# ---------------------------------------------------------------------------
def fetch_reviews_live(package_name, as_of=None):
    from google.oauth2 import service_account
    from googleapiclient.discovery import build
    creds = service_account.Credentials.from_service_account_file(
        "service_account.json", scopes=["https://www.googleapis.com/auth/androidpublisher"])
    api = build("androidpublisher", "v3", credentials=creds).reviews()
    out, token = [], None
    while True:
        resp = api.list(packageName=package_name, maxResults=100, token=token).execute()
        for r in resp.get("reviews", []):
            r["_packageName"] = package_name
            out.append(r)
        token = resp.get("tokenPagination", {}).get("nextPageToken")
        if not token:
            return out


if __name__ == "__main__":
    import os
    if os.path.exists(DB):
        os.remove(DB)          # POC: rebuild from scratch each run
    main()


In [ ]:
!python collector.py

## Step 3: noise, risk rules, multilingual topics

In [ ]:
%%writefile classify.py
"""
Step 3 - Filter noise, flag risk, classify every review (free, runs locally)
============================================================================

Three layers, in this order:

  A. NOISE FILTER (simple rules)
     too short / emoji only / gibberish / spam (links, "free coins")
     -> counted in volume stats, excluded from clustering. Spam is also
        marked "report to Google".

  B. RISK FLAGS (keyword rules, deliberately over-sensitive)
     child_safety, payment, legal, privacy, high_value
     Rules, not a model, because here missing one is worse than a false
     alarm. A flag forces the review to a human in step 6, whatever its category.

  C. CATEGORY (multilingual embeddings, no API, no cost)
     Every review becomes a vector ("a point on the map of meaning").
     Every category is described by a handful of example sentences.
     The review gets the category whose examples it is closest to.
     Works across languages: the examples are in English, the reviews are in
     Hebrew, German, Portuguese... the model maps them to the same place.
     Reviews close to no category -> "other" (or "off_topic" if they are
     closest to the off-topic examples).

Output (in reviews.db):
  review_analysis  one row per review: is_noise, noise_reason, category,
                   category_score, flags, rating_mismatch
  embeddings.npy + embedding_ids.json   vectors reused by step 4 (clustering)

Run in Colab:
  !pip install -q sentence-transformers
  then run this file (reviews.db and data/ must be next to it)
"""

import json
import re
import sqlite3
import unicodedata

import numpy as np

DB = "reviews.db"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # free, 50+ languages, ~470MB
MIN_SCORE = 0.35        # below this similarity -> "other"

# ---------------------------------------------------------------------------
# Taxonomy. "shared" applies to every app; genre families add their own.
# The example sentences are written independently of the fake data, in plain
# words a player would use. Adding a category = adding a few sentences.
# ---------------------------------------------------------------------------
TAXONOMY = {
    "shared": {
        "crash_stability": ["The app crashes when I open it", "Game closes by itself", "Stuck on a black screen",
                            "It freezes and shuts down", "Won't launch after updating"],
        "performance_battery": ["The game is slow and laggy", "My phone gets hot and the battery drains",
                                "Low frame rate, very choppy", "Takes too long to load"],
        "progress_account_loss": ["I lost all my progress", "My saved game was deleted", "My account was reset to level one",
                                  "Can't log in to my account anymore"],
        "payment_purchase": ["I paid but did not receive the item", "Money was charged but nothing arrived",
                             "Purchase failed and I want a refund", "I was charged twice"],
        "ads": ["Too many advertisements", "An ad after every level", "Ads that cannot be skipped",
                "I paid for no ads but still see ads"],
        "pricing_monetization": ["Items are too expensive", "Everything costs real money", "Pay to win",
                                 "Boosters cost too much"],
        "difficulty_balance": ["The levels are too hard", "I'm stuck on a level", "Lives recharge too slowly",
                               "Impossible without paying"],
        "content_repetitive": ["The levels feel repetitive", "Boring after a while", "Need more levels and content",
                               "I finished everything, nothing new"],
        "feature_request": ["Please add a new feature", "I wish there was an option to", "Would love more customization",
                            "Please add more options"],
        "praise": ["I love this game", "Great fun and relaxing", "Best game on my phone", "Beautiful graphics, highly recommend"],
    },
    "meta_world": {
        "chat_safety": ["A stranger sent private messages to my child", "An adult player is talking to kids in the chat",
                        "Someone asked my kid where they live", "The chat is not safe for children, please moderate"],
        "multiplayer_servers": ["Servers lag with many players", "Can't find or join my friends",
                                "Disconnected from the lobby", "Online mode is broken"],
    },
}
OFF_TOPIC_EXAMPLES = ["This review is about a different app", "Wrong app sorry", "Who won the football game",
                      "Does anyone know a good recipe", "What phone case should I buy"]
# "Love it, BUT it crashes": split mixed reviews and let the serious part win
CLAUSE_SPLIT = r"\b(?:but|however|although|aber|doch|jedoch|mas|porém|só que)\b|אבל|אך"
CRASH_WORDS = (r"crash|freez|closes|shuts down|won.t (open|start|launch)|stürzt|absturz|friert|schließt sich|"
               r"fecha sozinho|fecha toda|\btrava\b|não abre|קורס|קריסה|נסגר|לא נפתח")
PROGRESS_WORDS = r"progress|save|fortschritt|spielstand|progresso|התקדמות|השלבים"
LOSS_WORDS = r"lost|gone|wiped|deleted|reset|weg|gelöscht|verloren|perdi|apagou|sumiu|נמחק|מחק|נעלם|איבדתי"
SEVERE_CATEGORIES = {"crash_stability", "progress_account_loss", "payment_purchase", "chat_safety",
                     "performance_battery", "ads"}
PROTECTED_FLAGS = {"child_safety", "payment", "legal", "privacy", "store_policy"}
NEGATIVE_CATEGORIES = {"crash_stability", "progress_account_loss", "payment_purchase", "chat_safety"}

# ---------------------------------------------------------------------------
# Risk keywords (en / de / pt / he). Over-sensitive on purpose.
# ---------------------------------------------------------------------------
RISK = {
    "child_safety": [
        r"\b(my|our) (son|daughter|kid|child|children)\b.*\b(message|messag|chat|stranger|adult|address|live|lives)",
        r"\b(stranger|adult|grown.?up|creep|predator|groom)", r"\bnot safe for (kids|children)",
        r"\b(kids|children|minors)\b.*\b(messag|inappropriat|contact)", r"\binappropriat",
        r"\b(sohn|tochter|kind|kinder)\b.*\b(nachricht|chat|fremd|erwachsen|wohnt)", r"\b(fremde[rn]?|erwachsene[rn]?)\b",
        r"\b(filh[oa]|criança)\b.*\b(mensag|chat|estranh|adulto|mora)", r"\b(estranh[oa]|adulto)\b",
        r"(הבן|הבת|הילד|ילדים|לילדים).*(הודעות|צ'אט|זר|מבוגר|גר|גרה)", r"\b(זר|מבוגר)\b", r"לא בטוח לילדים",
    ],
    "payment": [
        # an act of paying + money symbols; NOT generic "costs money" (that is pricing, not a payment problem)
        r"\b(paid(?! only)|charged|purchased?|bought|refund|billing)\b", r"\b(bezahlt|abgebucht|gekauft|erstattung)",
        r"\b(paguei|cobrad|cobrou|cobraram|comprei|reembolso)", r"(שילמתי|חויבתי|קניתי|הכסף ירד|החזר כספי|רכישה)",
        r"[$€₪]|R\$",
    ],
    "legal": [r"\b(lawyer|sue|lawsuit|police|court|fraud|consumer protection)\b", r"\b(anwalt|polizei|betrug|klage|verbraucherschutz)",
              r"\b(advogado|polícia|processo|golpe|fraude|procon)", r"(עורך דין|תביעה|משטרה|הונאה|הגנת הצרכן)"],
    "privacy": [r"\b(personal data|my data|this data|privacy|gdpr|tracking me|tracking my)\b",
                r"\b(location|contacts)\b.*\b(kid|child|son|daughter)|\b(kid|child)\S*\b.*\b(location|contacts)\b",
                r"\b(datenschutz|meine daten|standort|kontakte)", r"\b(meus dados|privacidade|localização|contatos)",
                r"(פרטיות|המידע שלי|הנתונים שלי|מיקום|אנשי הקשר)"],
    # things that can get the app flagged by Google Play: ads policy, families policy, deceptive behavior
    "store_policy": [r"\b(casino|gambling|betting|adult content|sexual|nsfw)\b.*\bads?\b|\bads?\b.*\b(casino|gambling|betting|adult content|sexual)",
                     r"\b(installed|downloads?|downloaded) (another|other|an?) app", r"fake close|can.t close the ad|without asking",
                     r"(report|reporting|melde|denunciar|מדווח).{0,20}(google|play)", r"against google|google.s rules",
                     r"glücksspiel|cassino|apostas|הימורים|קזינו|תוכן למבוגרים", r"installiert|instalou outro|התקינה אפליקציה"],
    "high_value": [r"\bvip\b", r"\b(spent|spend|gastei|ausgegeben|gesteckt|הוצאתי)\b",
                   r"\b\d+ (years|jahren|anos)\b", r"(שנים|כבר \d+ שנ)", r"(\$|€|₪|R\$ ?)\s?\d{3,}|\d{3,}\s?(€|\$)"],
}
SPAM = [r"https?://|www\.|\.[a-z]{2,}\b(?<!\.\.)", r"\b(free|gratis|grátis) (coins|gems|münzen|moedas)",
        r"(מטבעות בחינם)", r"\b(hack|generator|unlimited)\b"]


# ---------------------------------------------------------------------------
# A. Noise
# ---------------------------------------------------------------------------
def letters(text):
    return [c for c in text if unicodedata.category(c).startswith("L")]


def noise_reason(text):
    t = text.strip()
    if not letters(t):
        return "emoji_or_symbols"
    if any(re.search(p, t.lower()) for p in SPAM):
        return "spam"
    words = re.findall(r"\w+", t)
    if len(words) <= 2 and len(t) < 15:
        return "too_short"
    low = t.lower()
    if len(set(low.split())) == 1 and len(low.split()) > 1:     # "lol lol lol"
        return "gibberish"
    letter_str = "".join(letters(low))
    if 5 <= len(letter_str) <= 15 and len(set(letter_str)) / len(letter_str) < 0.35:   # "aaaaaaa", "jjjjjj"
        return "gibberish"
    if re.fullmatch(r"[a-z]{5,}", letter_str) and not re.search(r"[aeiouy]", letter_str):   # "qwrtz"
        return "gibberish"
    if re.fullmatch(r"(asdf\w*|qwert\w*|zxcv\w*)", low):
        return "gibberish"
    return None


# ---------------------------------------------------------------------------
# B. Risk flags
# ---------------------------------------------------------------------------
def risk_flags(text, first_rating, rating):
    low = text.lower()
    flags = [name for name, pats in RISK.items() if any(re.search(p, low) for p in pats)]
    if first_rating is not None and first_rating >= 4 and rating <= 2:
        flags.append("rating_drop")        # was happy, came back angry
    return flags


# ---------------------------------------------------------------------------
# C. Embeddings + category
# ---------------------------------------------------------------------------
class Embedder:
    def __init__(self, name=MODEL_NAME):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(name)

    def encode(self, texts):
        return self.model.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True)


def build_prototypes(embedder, family):
    cats = {**TAXONOMY["shared"], **TAXONOMY.get(family, {})}
    names, centroids = [], []
    for name, examples in cats.items():
        v = embedder.encode(examples).mean(axis=0)
        names.append(name)
        centroids.append(v / np.linalg.norm(v))
    off = embedder.encode(OFF_TOPIC_EXAMPLES).mean(axis=0)
    return names, np.vstack(centroids), off / np.linalg.norm(off)


def run(embedder=None):
    embedder = embedder or Embedder()
    con = sqlite3.connect(DB)
    rows = con.execute("""SELECT r.review_id, r.text, r.star_rating, r.first_star_rating, a.genre_family
                          FROM reviews r JOIN apps a USING(package_name) ORDER BY r.first_seen_at""").fetchall()

    results, keep_idx = {}, []
    for i, (rid, text, rating, first, family) in enumerate(rows):
        flags = risk_flags(text, first, rating)
        # SAFETY RULE: a review with a risk flag is never thrown away as noise
        reason = None if set(flags) & PROTECTED_FLAGS else noise_reason(text)
        results[rid] = {"is_noise": int(reason is not None), "noise_reason": reason, "category": None,
                        "category_score": None, "secondary_category": None, "flags": flags,
                        "rating_mismatch": 0}
        if reason is None:
            keep_idx.append(i)

    print(f"Embedding {len(keep_idx):,} reviews (noise removed: {len(rows) - len(keep_idx):,})")
    vecs = embedder.encode([rows[i][1] for i in keep_idx])

    # mixed reviews: embed each clause separately too
    clauses = {}
    for i in keep_idx:
        raw = re.split(CLAUSE_SPLIT, rows[i][1], flags=re.IGNORECASE)
        parts = [p.strip(" ,.!") for p in raw if p and len(p.strip(" ,.!")) > 3]
        if len(raw) > 1 and parts:
            clauses[i] = parts
    flat = [p for parts in clauses.values() for p in parts]
    print(f"Mixed reviews split into clauses: {len(clauses):,}")
    clause_vecs = iter(embedder.encode(flat)) if flat else iter([])

    protos = {fam: build_prototypes(embedder, fam) for fam in {r[4] for r in rows}}
    for vec, i in zip(vecs, keep_idx):
        rid, text, rating, first, family = rows[i]
        names, centroids, off = protos[family]
        sims = centroids @ vec
        order = np.argsort(-sims)
        best, second = int(order[0]), int(order[1])
        res = results[rid]
        part_vecs = [next(clause_vecs) for _ in clauses.get(i, [])]
        if float(off @ vec) > sims[best] and not set(res["flags"]) & PROTECTED_FLAGS:
            res.update(is_noise=1, noise_reason="off_topic", category=None)
            continue
        cat, score = (names[best], float(sims[best])) if sims[best] >= MIN_SCORE else ("other", float(sims[best]))
        # severity wins: if any clause is clearly about a serious problem, that is the category
        for pv in part_vecs:
            ps = centroids @ pv
            pb = int(np.argmax(ps))
            if names[pb] in SEVERE_CATEGORIES and ps[pb] >= MIN_SCORE and cat not in SEVERE_CATEGORIES:
                cat, score = names[pb], float(ps[pb])
        # safety net for hidden bugs: explicit crash words beat praise/other, in any language
        low = text.lower()
        if cat in ("praise", "other", "content_repetitive", "feature_request", "performance_battery") \
                and re.search(CRASH_WORDS, low):
            cat = "crash_stability"
        # same idea for lost progress: data loss is too serious to leave to "closest example"
        if re.search(PROGRESS_WORDS, low) and re.search(LOSS_WORDS, low):
            cat = "progress_account_loss"
        res["category"], res["category_score"] = cat, round(score, 3)
        # second category when the review is genuinely about two things
        res["secondary_category"] = names[second] if (sims[second] >= MIN_SCORE and sims[best] - sims[second] < 0.05
                                                      and names[second] != cat) else None
        res["rating_mismatch"] = int(rating >= 4 and cat in NEGATIVE_CATEGORIES)

    con.execute("DROP TABLE IF EXISTS review_analysis")
    con.execute("""CREATE TABLE review_analysis (review_id TEXT PRIMARY KEY, is_noise INTEGER, noise_reason TEXT,
                   category TEXT, category_score REAL, secondary_category TEXT, flags TEXT,
                   rating_mismatch INTEGER)""")
    con.executemany("INSERT INTO review_analysis VALUES (?,?,?,?,?,?,?,?)",
                    [(rid, r["is_noise"], r["noise_reason"], r["category"], r["category_score"],
                      r["secondary_category"], ",".join(r["flags"]), r["rating_mismatch"]) for rid, r in results.items()])
    con.commit()

    ids = [rows[i][0] for i in keep_idx]
    np.save("embeddings.npy", vecs.astype(np.float32))
    json.dump(ids, open("embedding_ids.json", "w"))
    report(con)
    con.close()


# ---------------------------------------------------------------------------
def report(con):
    q = lambda s: con.execute(s).fetchall()
    total = q("SELECT COUNT(*) FROM review_analysis")[0][0]
    print(f"\n=== {total:,} reviews ===")
    print("\nNoise (counted, not clustered):")
    for reason, n in q("SELECT noise_reason, COUNT(*) FROM review_analysis WHERE is_noise=1 GROUP BY 1 ORDER BY 2 DESC"):
        print(f"  {reason:18s} {n:,}")
    print("\nCategories by app (% of that app's non-noise reviews):")
    data = q("""SELECT a.app_name, ra.category, COUNT(*) FROM review_analysis ra JOIN reviews r USING(review_id)
                JOIN apps a USING(package_name) WHERE ra.is_noise=0 GROUP BY 1, 2""")
    apps = sorted({d[0] for d in data})
    for app in apps:
        tot = sum(n for a, _, n in data if a == app)
        top = sorted([(n, c) for a, c, n in data if a == app], reverse=True)[:6]
        print(f"  {app:13s} " + "  ".join(f"{c} {n / tot:.0%}" for n, c in top))
    print("\nRisk flags:")
    for flag in ["child_safety", "payment", "legal", "privacy", "high_value", "rating_drop"]:
        n = q(f"SELECT COUNT(*) FROM review_analysis WHERE ',' || flags || ',' LIKE '%,{flag},%'")[0][0]
        print(f"  {flag:14s} {n:,}")
    print(f"\nRating mismatch (4-5 stars, text reports a serious problem): "
          f"{q('SELECT COUNT(*) FROM review_analysis WHERE rating_mismatch=1')[0][0]}")


if __name__ == "__main__":
    run()


In [ ]:
!python classify.py

## Step 4: themes from the stream

In [ ]:
%%writefile cluster.py
"""
Step 4 - Discover themes and keep learning from the stream
==========================================================

Step 3 gave every review a FIXED category (crash, ads, feature_request...).
Step 4 finds the SPECIFIC themes inside each category, per genre, without us
telling it what to look for. "feature_request" in PeleGame splits into e.g.
"avatar customization" vs "pets"; "crash_stability" in Merge Meadow becomes
"crashes on launch after update".

How it learns from the stream (instead of re-reading everything every day):

  1. BASELINE  - cluster the first 30 days (HDBSCAN on the step-3 embeddings,
                 separately for every genre x category).
  2. DAILY     - each new review joins the closest existing theme, if it is
                 close enough (each theme learns its own "how close is close"
                 from its members).
  3. NEW THEME - reviews that fit no theme wait in an "unassigned pool".
                 When enough similar ones pile up, the pool is clustered and a
                 NEW THEME is born, with the date it appeared. That date is the
                 "something new started happening" alert.

Naming is free too: each theme gets its top keywords (across all languages)
and its most typical review as a quote. An LLM could write nicer names later;
it is optional, not required.

Output (in reviews.db):
  themes          one row per theme: genre, category, keywords, quote, born_on, size
  review_themes   review_id -> theme_id (or NULL = still unassigned), assigned_on

Run after classify.py:  python cluster.py
"""

import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import re
import sqlite3
from collections import Counter, defaultdict
from datetime import date, timedelta

import numpy as np
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer

DB = "reviews.db"
BASELINE_DAYS = 30
MIN_THEME = 12            # smallest group we call a theme
NEW_THEME_WINDOW = 7      # pool reviews older than this don't count toward a new theme
MIN_SIM_FLOOR = 0.45      # never attach a review below this similarity
MERGE_SIM = 0.80          # two themes this similar are the same theme (e.g. same complaint in 2 languages)

STOP = set("""the a an and or but to of in on for is it this that i my me we you be are was with so at as
very please just all can not no app game der die das und ist ich es nicht ein eine zu mit mein meine sehr
o a os as e de do da que em um uma para com não eu meu minha muito jogo את של על זה לא אני עם גם כל הוא
היא יותר מדי בבקשה honestly hi devs ehrlich gesagt hallo sinceramente oi האמת היי""".split())


# ---------------------------------------------------------------------------
def load():
    con = sqlite3.connect(DB)
    vecs = np.load("embeddings.npy")
    ids = json.load(open("embedding_ids.json"))
    pos = {rid: i for i, rid in enumerate(ids)}
    rows = con.execute("""
        SELECT r.review_id, substr(r.last_modified_at, 1, 10), r.text, r.reviewer_language,
               r.app_version_name, r.device, a.genre, a.app_name, ra.category
        FROM reviews r JOIN apps a USING(package_name) JOIN review_analysis ra USING(review_id)
        WHERE ra.is_noise = 0 ORDER BY r.last_modified_at""").fetchall()
    reviews = [dict(zip(["id", "day", "text", "lang", "version", "device", "genre", "app", "category"], r))
               for r in rows if r[0] in pos]
    for r in reviews:
        r["vec"] = vecs[pos[r["id"]]]
    return con, reviews


def unit(v):
    return v / (np.linalg.norm(v) + 1e-9)


class Theme:
    _next = 0

    def __init__(self, genre, category, members, born_on):
        Theme._next += 1
        self.id = f"T{Theme._next:03d}"
        self.genre, self.category, self.born_on = genre, category, born_on
        self.members = list(members)
        self.refresh()

    def refresh(self):
        m = np.vstack([r["vec"] for r in self.members])
        self.centroid = unit(m.mean(axis=0))
        sims = m @ self.centroid
        # "how close is close" for this theme: the 5th percentile of its own members
        self.threshold = max(MIN_SIM_FLOOR, float(np.percentile(sims, 5)))

    def add(self, r):
        self.members.append(r)
        if len(self.members) % 25 == 0:      # drift slowly with the stream
            self.refresh()


def discover(group, genre, category, born_on):
    """HDBSCAN on a set of reviews -> list of new Themes (+ leftovers)."""
    if len(group) < MIN_THEME:
        return [], group
    X = np.vstack([r["vec"] for r in group])
    labels = HDBSCAN(min_cluster_size=MIN_THEME, min_samples=5, metric="euclidean").fit_predict(X)
    themes = [Theme(genre, category, [r for r, l in zip(group, labels) if l == k], born_on)
              for k in sorted(set(labels) - {-1})]
    leftovers = [r for r, l in zip(group, labels) if l == -1]
    return themes, leftovers


def merge_similar(themes):
    """Join themes of the same genre x category whose centres are almost identical.
    Keeps the earliest born_on, so a merged theme is not reported as 'new'."""
    merged = True
    while merged:
        merged = False
        for i, a in enumerate(themes):
            for b in themes[i + 1:]:
                if a.genre == b.genre and a.category == b.category and float(a.centroid @ b.centroid) >= MERGE_SIM:
                    keep, drop = (a, b) if len(a.members) >= len(b.members) else (b, a)
                    keep.members += drop.members
                    keep.born_on = min(keep.born_on, drop.born_on)
                    keep.refresh()
                    themes.remove(drop)
                    merged = True
                    break
            if merged:
                break
    return themes


def name_themes(themes):
    """Keywords via c-TF-IDF (one 'document' per theme) + the most typical review."""
    docs = [" ".join(r["text"] for r in t.members) for t in themes]
    tok = lambda s: [w for w in re.findall(r"[^\W\d_]{3,}", s.lower()) if w not in STOP]
    tfidf = TfidfVectorizer(tokenizer=tok, lowercase=False, token_pattern=None, sublinear_tf=True)
    M = tfidf.fit_transform(docs)
    vocab = np.array(tfidf.get_feature_names_out())
    for i, t in enumerate(themes):
        row = M[i].toarray()[0]
        t.keywords = ", ".join(vocab[np.argsort(-row)[:6]])
        sims = np.vstack([r["vec"] for r in t.members]) @ t.centroid
        order = np.argsort(-sims)
        en = [t.members[j] for j in order[:20] if t.members[j]["lang"] == "en"]
        t.quote = (en[0] if en else t.members[order[0]])["text"]


# ---------------------------------------------------------------------------
def run():
    con, reviews = load()
    start = date.fromisoformat(reviews[0]["day"])
    cutoff = (start + timedelta(days=BASELINE_DAYS)).isoformat()
    themes, assigned, pool = [], {}, defaultdict(list)
    births = []

    # 1. BASELINE
    base = defaultdict(list)
    for r in reviews:
        if r["day"] < cutoff:
            base[(r["genre"], r["category"])].append(r)
    for (genre, cat), group in base.items():
        new, left = discover(group, genre, cat, born_on=reviews[0]["day"])
        themes += new
        for t in new:
            for r in t.members:
                assigned[r["id"]] = (t.id, r["day"])
        pool[(genre, cat)] += left
    raw = len(themes)
    themes = merge_similar(themes)
    for t in themes:
        for r in t.members:
            assigned[r["id"]] = (t.id, r["day"])
    print(f"Baseline ({BASELINE_DAYS} days): {raw} raw groups -> {len(themes)} themes after merging look-alikes")

    # 2 + 3. DAILY STREAM
    by_day = defaultdict(list)
    for r in reviews:
        if r["day"] >= cutoff:
            by_day[r["day"]].append(r)
    for day in sorted(by_day):
        for r in by_day[day]:
            cands = [t for t in themes if t.genre == r["genre"] and t.category == r["category"]]
            best = max(cands, key=lambda t: float(r["vec"] @ t.centroid), default=None)
            if best is not None and float(r["vec"] @ best.centroid) >= best.threshold:
                best.add(r)
                assigned[r["id"]] = (best.id, day)
            else:
                pool[(r["genre"], r["category"])].append(r)
        # try to form new themes from recent unassigned reviews
        window = (date.fromisoformat(day) - timedelta(days=NEW_THEME_WINDOW)).isoformat()
        for key, waiting in pool.items():
            recent = [r for r in waiting if r["day"] > window]
            if len(recent) < MIN_THEME:
                continue
            new, _ = discover(recent, key[0], key[1], born_on=day)
            new = merge_similar(new)
            for t in new:
                # a "new" group that looks like an existing theme just joins it
                twin = max((x for x in themes if x.genre == t.genre and x.category == t.category),
                           key=lambda x: float(x.centroid @ t.centroid), default=None)
                if twin is not None and float(twin.centroid @ t.centroid) >= MERGE_SIM:
                    for r in t.members:
                        twin.add(r)
                        assigned[r["id"]] = (twin.id, day)
                    continue
                themes.append(t)
                births.append(t)
                for r in t.members:
                    assigned[r["id"]] = (t.id, day)
            if new:
                taken = {r["id"] for t in new for r in t.members}
                pool[key] = [r for r in waiting if r["id"] not in taken]

    name_themes(themes)

    # save
    con.execute("DROP TABLE IF EXISTS themes")
    con.execute("DROP TABLE IF EXISTS review_themes")
    con.execute("""CREATE TABLE themes (theme_id TEXT PRIMARY KEY, genre TEXT, category TEXT, keywords TEXT,
                   quote TEXT, born_on TEXT, size INTEGER, languages TEXT, apps TEXT)""")
    con.execute("CREATE TABLE review_themes (review_id TEXT PRIMARY KEY, theme_id TEXT, assigned_on TEXT)")
    for t in themes:
        langs = Counter(r["lang"] for r in t.members)
        apps = Counter(r["app"] for r in t.members)
        con.execute("INSERT INTO themes VALUES (?,?,?,?,?,?,?,?,?)",
                    (t.id, t.genre, t.category, t.keywords, t.quote, t.born_on, len(t.members),
                     json.dumps(dict(langs)), json.dumps(dict(apps))))
    con.executemany("INSERT INTO review_themes VALUES (?,?,?)",
                    [(r["id"], *assigned.get(r["id"], (None, None))) for r in reviews])
    con.commit()

    # report
    n_assigned = sum(1 for r in reviews if r["id"] in assigned)
    print(f"Final: {len(themes)} themes, {n_assigned:,}/{len(reviews):,} reviews in a theme "
          f"({n_assigned / len(reviews):.0%}), rest still unassigned")
    print("\nNEW THEMES born from the stream (after the baseline):")
    for t in sorted(births, key=lambda t: t.born_on):
        print(f"  {t.born_on}  {t.id}  {t.genre:13s} {t.category:22s} n={len(t.members):<4} [{t.keywords}]")
        print(f"              \"{t.quote[:90]}\"")
    print("\nLargest themes:")
    for t in sorted(themes, key=lambda t: -len(t.members))[:12]:
        langs = Counter(r["lang"] for r in t.members)
        print(f"  {t.id} {t.genre:13s} {t.category:22s} n={len(t.members):<5} langs={dict(langs)}")
        print(f"        [{t.keywords}]  \"{t.quote[:80]}\"")
    con.close()


if __name__ == "__main__":
    run()


In [ ]:
!python cluster.py

## Step 5: trends and alerts

In [ ]:
%%writefile trends.py
"""
Step 5 - Trends, normalization and alerts
=========================================

Turns the classified stream into a short list of things worth acting on.

Normalization (why a small app can outrank a big one):
  share = reviews about X / all real reviews of that app (same days)
  lift  = share now / the app's own normal share
  An alert needs a high lift AND enough reviews to not be luck (binomial
  z-score + a minimum count). The absolute count is still shown, so a
  product person also sees how many players are affected.

Alert types:
  SAFETY        any child-safety / legal report -> immediately, regardless of volume
  RELEASE       a category jumps in a new app version vs the previous version
                (+ which devices it is concentrated on)
  SPIKE         a category's 7-day share vs the app's previous 4 weeks
  NEW_THEME     a theme that did not exist before was born from the stream (step 4)
  EMERGING      slow steady growth over weeks (a feature request gaining traction)
  CHRONIC       always high, no spike - a known issue for the roadmap, not an incident

The script replays the 60 days as if it ran every morning, so each alert has
the day it would FIRST have been raised. That date is the detection speed.

Output (in reviews.db):
  daily_metrics  app x category x day: count, app total, share
  alerts         one row per issue: type, app, category, first_raised, last_seen,
                 evidence (lift, z, counts, version, devices), priority

Run after cluster.py:  python trends.py
"""

import json
import math
import sqlite3
from datetime import date, timedelta

import pandas as pd

DB = "reviews.db"
RECENT = 7            # days in the "now" window
BASELINE = 28         # days in the "normal" window before it
MIN_COUNT = 15        # never alert on fewer reviews than this
MIN_LIFT = 2.0
MIN_Z = 3.0
NOT_PROBLEMS = {"praise", "other"}          # we don't alert on more praise


def z_score(k, n, p0):
    """How surprising is k out of n if the normal rate is p0 (binomial)."""
    p0 = min(max(p0, 1e-3), 0.999)
    return (k - n * p0) / math.sqrt(n * p0 * (1 - p0)) if n else 0.0


def load(con):
    df = pd.read_sql("""
        SELECT r.review_id, substr(r.last_modified_at, 1, 10) AS day, a.app_name AS app, a.genre,
               r.app_version_name AS version, r.device_name AS device, r.star_rating AS rating,
               ra.category, ra.flags, ra.is_noise, rt.theme_id
        FROM reviews r JOIN apps a USING(package_name) JOIN review_analysis ra USING(review_id)
        LEFT JOIN review_themes rt USING(review_id)""", con)
    df["day"] = pd.to_datetime(df["day"])
    return df


# ---------------------------------------------------------------------------
def detect(df, themes, as_of, history_start):
    """All alerts that would be raised on the morning of `as_of` (using data up to the day before)."""
    alerts = []
    real = df[(df.is_noise == 0) & (df.day < as_of)]
    recent = real[real.day >= as_of - pd.Timedelta(days=RECENT)]
    base = real[(real.day < as_of - pd.Timedelta(days=RECENT)) &
                (real.day >= as_of - pd.Timedelta(days=RECENT + BASELINE))]

    # SAFETY: volume does not matter
    for flag in ("child_safety", "legal"):
        hits = df[(df.day < as_of) & (df.day >= as_of - pd.Timedelta(days=1)) & df["flags"].str.contains(flag, na=False)]
        for app, g in hits.groupby("app"):
            alerts.append(dict(type="SAFETY", app=app, category=flag, key=f"SAFETY|{app}|{flag}",
                               evidence=dict(new_reports=len(g), review_ids=list(g.review_id))))

    if as_of < history_start + pd.Timedelta(days=RECENT + 14):
        return alerts            # not enough history yet for "normal" - only safety runs
    for app, g in recent.groupby("app"):
        n_now = len(g)
        b = base[base.app == app]
        n_base = len(b)
        if n_now < 30 or n_base < 100:
            continue
        for cat, k in g.category.value_counts().items():
            if cat in NOT_PROBLEMS or k < MIN_COUNT:
                continue
            p_now = k / n_now
            p_base = (b.category == cat).sum() / n_base
            lift = (p_now + 0.005) / (p_base + 0.005)            # smoothed: new categories don't divide by 0
            z = z_score(k, n_now, p_base)
            if lift >= MIN_LIFT and z >= MIN_Z:
                alerts.append(dict(type="SPIKE", app=app, category=cat, key=f"SPIKE|{app}|{cat}",
                                   evidence=dict(count_7d=int(k), share_now=round(p_now, 3),
                                                 share_normal=round(p_base, 3), lift=round(lift, 1), z=round(z, 1))))

    # RELEASE: newest version vs previous version, same app
    for app, g in real[real.day >= as_of - pd.Timedelta(days=RECENT + BASELINE)].groupby("app"):
        versions = sorted(g.version.dropna().unique(), key=lambda v: [int(x) for x in v.split(".")])
        if len(versions) < 2:
            continue
        new_v, old_v = versions[-1], versions[-2]
        new, old = g[g.version == new_v], g[g.version == old_v]
        if len(new) < 30 or len(old) < 100:
            continue
        for cat, k in new.category.value_counts().items():
            if cat in NOT_PROBLEMS or k < MIN_COUNT:
                continue
            p_new, p_old = k / len(new), (old.category == cat).sum() / len(old)
            lift, z = (p_new + 0.005) / (p_old + 0.005), z_score(k, len(new), p_old)
            if lift >= MIN_LIFT and z >= MIN_Z:
                # where is it concentrated? device share inside the problem vs in the whole version
                prob = new[new.category == cat]
                dev = (prob.device.value_counts(normalize=True) /
                       new.device.value_counts(normalize=True)).dropna().sort_values(ascending=False)
                hot = [f"{d} ({prob.device.value_counts(normalize=True)[d]:.0%} of reports, x{x:.1f})"
                       for d, x in dev.items() if x >= 1.5][:4]
                alerts.append(dict(type="RELEASE", app=app, category=cat, key=f"RELEASE|{app}|{cat}|{new_v}",
                                   evidence=dict(version=new_v, previous=old_v, count=int(k), share_new=round(p_new, 3),
                                                 share_previous=round(p_old, 3), lift=round(lift, 1), z=round(z, 1),
                                                 concentrated_on=hot)))

    # NEW_THEME: themes born yesterday (after the baseline period)
    born = themes[(themes.born_on == (as_of - pd.Timedelta(days=1)).strftime("%Y-%m-%d")) &
                  (themes.born_on > history_start.strftime("%Y-%m-%d"))]       # baseline themes aren't "new"
    for t in born.itertuples():
        if t.category in NOT_PROBLEMS:
            continue
        apps = json.loads(t.apps)
        alerts.append(dict(type="NEW_THEME", app=max(apps, key=apps.get), category=t.category, key=f"NEW_THEME|{t.theme_id}",
                           evidence=dict(theme=t.theme_id, size_at_birth=int(t.size), keywords=t.keywords, quote=t.quote)))
    return alerts


def slow_trends(df):
    """EMERGING and CHRONIC need the whole history, so they are computed once, weekly."""
    alerts = []
    real = df[df.is_noise == 0].copy()
    real["week"] = (real.day - real.day.min()).dt.days // 7
    weeks = sorted(real.week.unique())[:-1] or [0]          # drop the last, partial week
    real = real[real.week.isin(weeks)]
    for app, g in real.groupby("app"):
        tot = g.groupby("week").size()
        for cat in g.category.dropna().unique():
            if cat in NOT_PROBLEMS:
                continue
            share = (g[g.category == cat].groupby("week").size() / tot).reindex(tot.index, fill_value=0)
            if len(share) < 4:
                continue
            first, last = share.iloc[:2].mean(), share.iloc[-2:].mean()
            rising_weeks = sum(b > a for a, b in zip(share, share.iloc[1:]))
            if last >= 2 * max(first, 0.01) and rising_weeks >= len(share) * 0.6 and last >= 0.05:
                alerts.append(dict(type="EMERGING", app=app, category=cat, key=f"EMERGING|{app}|{cat}",
                                   evidence=dict(share_first_weeks=round(first, 3), share_last_weeks=round(last, 3),
                                                 weekly_share=[round(x, 3) for x in share])))
            elif share.min() >= 0.10 and share.max() / max(share.min(), 1e-3) < 1.8:
                alerts.append(dict(type="CHRONIC", app=app, category=cat, key=f"CHRONIC|{app}|{cat}",
                                   evidence=dict(avg_share=round(share.mean(), 3),
                                                 weekly_share=[round(x, 3) for x in share])))
    return alerts


PRIORITY = {"SAFETY": 100, "RELEASE": 80, "SPIKE": 60, "NEW_THEME": 50, "EMERGING": 30, "CHRONIC": 20}
SEVERE = {"child_safety", "legal", "payment_purchase", "progress_account_loss", "crash_stability", "chat_safety"}


def priority(a):
    p = PRIORITY[a["type"]] + (15 if a["category"] in SEVERE else 0)
    ev = a["evidence"]
    p += min(15, int(ev.get("lift", 0)))
    return p


def run():
    con = sqlite3.connect(DB)
    df = load(con)
    themes = pd.read_sql("SELECT * FROM themes", con)
    start, end = df.day.min(), df.day.max()

    # daily metrics (for charts)
    real = df[df.is_noise == 0]
    daily = real.groupby(["app", "category", "day"]).size().rename("count").reset_index()
    totals = real.groupby(["app", "day"]).size().rename("app_total").reset_index()
    daily = daily.merge(totals, on=["app", "day"])
    daily["share"] = (daily["count"] / daily["app_total"]).round(4)
    daily["day"] = daily.day.dt.strftime("%Y-%m-%d")
    daily.to_sql("daily_metrics", con, if_exists="replace", index=False)

    # replay every morning
    issues = {}
    day = start + pd.Timedelta(days=1)        # safety works from day 1; trend checks wait for enough history
    while day <= end + pd.Timedelta(days=1):
        for a in detect(df, themes, day, start):
            k = a["key"]
            if k not in issues:
                issues[k] = {**a, "first_raised": day.strftime("%Y-%m-%d")}
            issues[k]["last_seen"] = day.strftime("%Y-%m-%d")
            if a["type"] != "SAFETY":
                issues[k]["evidence"] = a["evidence"]          # keep the latest numbers
        day += pd.Timedelta(days=1)
    for a in slow_trends(df):
        issues[a["key"]] = {**a, "first_raised": end.strftime("%Y-%m-%d"), "last_seen": end.strftime("%Y-%m-%d")}

    # safety: replace evidence with the full list
    for k, a in issues.items():
        if a["type"] == "SAFETY":
            hits = df[(df.app == a["app"]) & df["flags"].str.contains(a["category"], na=False)]
            a["evidence"] = dict(total_reports=len(hits), first=hits.day.min().strftime("%Y-%m-%d"),
                                 review_ids=list(hits.review_id))

    rows = []
    for a in issues.values():
        a["priority"] = priority(a)
        rows.append((a["key"], a["type"], a["app"], a["category"], a["first_raised"], a["last_seen"],
                     a["priority"], json.dumps(a["evidence"], ensure_ascii=False)))
    con.execute("DROP TABLE IF EXISTS alerts")
    con.execute("""CREATE TABLE alerts (alert_key TEXT PRIMARY KEY, type TEXT, app TEXT, category TEXT,
                   first_raised TEXT, last_seen TEXT, priority INTEGER, evidence TEXT)""")
    con.executemany("INSERT INTO alerts VALUES (?,?,?,?,?,?,?,?)", rows)
    con.commit()

    # ROLL UP: one issue per app x category. A product person should see
    # "Merge Meadow crashes on 2.4.0" once, with all the signals that point to it.
    grouped = {}
    for a in issues.values():
        grouped.setdefault((a["app"], a["category"]), []).append(a)
    roll = []
    for (app, cat), sigs in grouped.items():
        sigs.sort(key=lambda a: (-a["priority"], a["first_raised"]))
        lead = sigs[0]
        types = sorted({a["type"] for a in sigs}, key=lambda t: -PRIORITY[t])
        roll.append(dict(app=app, category=cat, lead=lead["type"], first_raised=min(a["first_raised"] for a in sigs),
                         priority=lead["priority"] + 5 * (len(types) - 1),     # more independent signals = more sure
                         signals=[f"{a['type']} {a['first_raised']}" for a in sorted(sigs, key=lambda a: a["first_raised"])],
                         themes=[a["evidence"]["theme"] for a in sigs if a["type"] == "NEW_THEME"],
                         evidence=lead["evidence"], types=types))
    con.execute("DROP TABLE IF EXISTS issues")
    con.execute("""CREATE TABLE issues (app TEXT, category TEXT, lead_signal TEXT, first_raised TEXT, priority INTEGER,
                   signals TEXT, themes TEXT, evidence TEXT)""")
    con.executemany("INSERT INTO issues VALUES (?,?,?,?,?,?,?,?)",
                    [(r["app"], r["category"], r["lead"], r["first_raised"], r["priority"], json.dumps(r["signals"]),
                      json.dumps(r["themes"]), json.dumps(r["evidence"], ensure_ascii=False)) for r in roll])
    con.commit()

    print(f"{len(rows)} signals -> {len(roll)} issues over {(end - start).days + 1} days\n")
    for r in sorted(roll, key=lambda r: (-r["priority"], r["first_raised"])):
        a = {"type": r["lead"], "evidence": r["evidence"], "priority": r["priority"], "first_raised": r["first_raised"],
             "app": r["app"], "category": r["category"]}
        ev = a["evidence"]
        if a["type"] == "SAFETY":
            detail = f"{ev['total_reports']} reports since {ev['first']}"
        elif a["type"] == "RELEASE":
            detail = (f"v{ev['version']}: {ev['share_new']:.0%} of reviews vs {ev['share_previous']:.0%} in "
                      f"v{ev['previous']} (x{ev['lift']}, n={ev['count']})  devices: {', '.join(ev['concentrated_on']) or '-'}")
        elif a["type"] == "SPIKE":
            detail = f"{ev['share_now']:.0%} of last 7 days vs normal {ev['share_normal']:.0%} (x{ev['lift']}, n={ev['count_7d']})"
        elif a["type"] == "NEW_THEME":
            detail = f"{ev['theme']} [{ev['keywords']}] \"{ev['quote'][:60]}\""
        elif a["type"] == "EMERGING":
            detail = f"weekly share {ev['share_first_weeks']:.0%} -> {ev['share_last_weeks']:.0%}"
        else:
            detail = f"steady ~{ev['avg_share']:.0%} of reviews every week"
        print(f"  [{a['priority']:3d}] {a['first_raised']}  {a['app']:13s} {a['category']:22s} {a['type']:9s} {detail}")
        extra = [t for t in r["types"] if t != r["lead"]]
        if extra or r["themes"]:
            print(f"        also: {', '.join(extra) or '-'}"
                  + (f"   themes: {', '.join(r['themes'])}" if r["themes"] else ""))
    con.close()


if __name__ == "__main__":
    run()


In [ ]:
!python trends.py

## Step 6: triage, replies, policy guard

In [ ]:
%%writefile triage.py
"""
Step 6 - Triage: who answers what, and safe automatic replies
=============================================================

Every review goes to exactly ONE lane. The lanes are checked from the most
serious down, so a review can never "fall" into automation if any serious
signal is present.

  1. SAFETY       child safety / chat safety     -> Trust & Safety + Legal notified now,
                                                    a person replies (SLA 2h). Never automatic.
  2. LEGAL        legal threat / privacy           -> Legal notified, a person replies (SLA 24h)
  3. HUMAN_VIP    money, lost progress, high-value player, 5->1 rating drop
                                                  -> support agent, with a draft to edit (SLA 24h)
  4. HUMAN_CHECK  the system is unsure (low score / angry but unclassified)
                                                  -> person decides (SLA 48h)
  5. INCIDENT     review about an active release issue (step 5)
                                                  -> one approved incident reply, edited again when
                                                     the fix ships (Play lets you edit a reply)
  6. AUTO         praise, feature ideas, known chronic topics
                                                  -> template reply in the reviewer's language
  7. NO_REPLY     noise (counted only); spam is reported to Google

Every automatic reply must pass the POLICY GUARD below, or it is not posted
and goes to a person instead. 5% of automatic replies are also sampled for a
human quality check.

Policy source: Google Play "Ratings, Reviews, and Installs" policy - don't
ask for a higher rating, no rewards for ratings, no coupons / codes / links /
email addresses in review content, point users to support resources.
API limits: 350 characters, plain text, 2,000 replies per app per day.

Output (in reviews.db):
  triage         review_id, lane, reason, priority, team, sla_hours, draft, reply_status, violations
  notifications  team, review_id, sent_on, message   (who was told about what)
  reviews.developer_reply / replied_at   filled for replies that were "posted"

Run after trends.py:  python triage.py
"""

import json
import random
import re
import sqlite3
from collections import Counter, defaultdict

DB = "reviews.db"
SUPPORT_URL = "support.example.com"       # the only link a reply may contain
MAX_LEN = 350
DAILY_QUOTA = 2000
QA_SAMPLE = 0.05
LOW_CONFIDENCE = 0.40
rng = random.Random(1)

# ---------------------------------------------------------------------------
# Approved templates (written once, reviewed by legal/brand, in 4 languages)
# ---------------------------------------------------------------------------
T = {
    "praise": {
        "en": "Thank you so much for playing {app}! We're really happy you're enjoying it.",
        "de": "Vielen Dank, dass du {app} spielst! Wir freuen uns sehr, dass es dir gefällt.",
        "pt": "Muito obrigado por jogar {app}! Ficamos muito felizes que você esteja curtindo.",
        "he": "תודה רבה שאתם משחקים ב־{app}! אנחנו ממש שמחים שאתם נהנים.",
    },
    "feature_request": {
        "en": "Thanks for the idea! We've shared it with the {app} team, player suggestions really shape what we build next.",
        "de": "Danke für die Idee! Wir haben sie an das {app}-Team weitergegeben. Vorschläge von Spielern prägen, was wir als Nächstes bauen.",
        "pt": "Obrigado pela ideia! Já compartilhamos com a equipe do {app}, as sugestões dos jogadores ajudam a definir o que vem a seguir.",
        "he": "תודה על הרעיון! העברנו אותו לצוות של {app}. ההצעות של השחקנים באמת משפיעות על מה שנבנה בהמשך.",
    },
    "ads": {
        "en": "Thanks for the honest feedback about ads. We're reviewing how often they appear. If you bought ad removal and still see ads, please contact us at " + SUPPORT_URL + " so we can fix it.",
        "de": "Danke für dein ehrliches Feedback zur Werbung. Wir prüfen gerade, wie oft sie erscheint. Falls du Werbefreiheit gekauft hast und trotzdem Werbung siehst, melde dich bitte unter " + SUPPORT_URL + ".",
        "pt": "Obrigado pelo feedback sincero sobre os anúncios. Estamos revisando a frequência deles. Se você comprou a remoção de anúncios e ainda os vê, fale conosco em " + SUPPORT_URL + ".",
        "he": "תודה על המשוב הכן לגבי הפרסומות. אנחנו בודקים את התדירות שלהן. אם רכשתם הסרת פרסומות ועדיין רואים אותן, פנו אלינו ב־" + SUPPORT_URL + ".",
    },
    "difficulty_balance": {
        "en": "Thanks for letting us know! Level balance is something we keep tuning, and your feedback goes straight to the design team.",
        "de": "Danke für dein Feedback! Wir passen die Schwierigkeit der Level laufend an, und deine Rückmeldung geht direkt an das Design-Team.",
        "pt": "Obrigado por avisar! Estamos sempre ajustando a dificuldade das fases, e seu feedback vai direto para a equipe de design.",
        "he": "תודה שסיפרתם לנו! אנחנו ממשיכים לכוונן את רמת הקושי של השלבים, והמשוב שלכם עובר ישירות לצוות העיצוב.",
    },
    "pricing_monetization": {
        "en": "Thanks for the feedback on prices. We want {app} to feel fair for everyone, and we've passed your comments to the team.",
        "de": "Danke für dein Feedback zu den Preisen. {app} soll sich für alle fair anfühlen, deshalb haben wir deine Hinweise ans Team weitergegeben.",
        "pt": "Obrigado pelo feedback sobre os preços. Queremos que {app} seja justo para todos e repassamos seus comentários à equipe.",
        "he": "תודה על המשוב לגבי המחירים. חשוב לנו ש־{app} ירגיש הוגן לכולם, והעברנו את ההערות שלכם לצוות.",
    },
    "content_repetitive": {
        "en": "Thanks for playing so much of {app}! New content is on the way, and your feedback helps us decide what comes next.",
        "de": "Danke, dass du so viel {app} spielst! Neue Inhalte sind unterwegs, und dein Feedback hilft uns bei der Planung.",
        "pt": "Obrigado por jogar tanto {app}! Novos conteúdos estão a caminho e seu feedback nos ajuda a decidir o que vem depois.",
        "he": "תודה שאתם משחקים כל כך הרבה ב־{app}! תוכן חדש בדרך, והמשוב שלכם עוזר לנו להחליט מה יגיע בהמשך.",
    },
    "performance_battery": {
        "en": "Sorry about the performance trouble. Updating to the latest version and closing other apps can help. If it continues, please write to " + SUPPORT_URL + " with your device model.",
        "de": "Entschuldige die Probleme mit der Leistung. Ein Update auf die neueste Version und das Schließen anderer Apps kann helfen. Sonst schreib uns bitte an " + SUPPORT_URL + ".",
        "pt": "Desculpe pelos problemas de desempenho. Atualizar para a versão mais recente e fechar outros apps pode ajudar. Se continuar, escreva para " + SUPPORT_URL + ".",
        "he": "מצטערים על בעיות הביצועים. עדכון לגרסה האחרונה וסגירת אפליקציות אחרות יכולים לעזור. אם זה ממשיך, כתבו לנו ב־" + SUPPORT_URL + ".",
    },
    "multiplayer_servers": {
        "en": "Sorry about the lag. We're working on server capacity for busy hours. If you keep getting disconnected, please let us know at " + SUPPORT_URL + ".",
        "de": "Entschuldige die Verzögerungen. Wir arbeiten an mehr Serverkapazität für Stoßzeiten. Wenn du oft getrennt wirst, melde dich bitte unter " + SUPPORT_URL + ".",
        "pt": "Desculpe pela lentidão. Estamos aumentando a capacidade dos servidores nos horários de pico. Se continuar caindo, fale conosco em " + SUPPORT_URL + ".",
        "he": "מצטערים על האיטיות. אנחנו מגדילים את קיבולת השרתים בשעות העומס. אם החיבור ממשיך להתנתק, ספרו לנו ב־" + SUPPORT_URL + ".",
    },
    "incident_fixed": {
        "en": "Update: the fix is out in version {fixed}. Please update the game from Google Play. If you still have trouble, write to us at " + SUPPORT_URL + ".",
        "de": "Update: Die Lösung ist in Version {fixed} verfügbar. Bitte aktualisiere das Spiel über Google Play. Falls es weiter Probleme gibt, schreib uns an " + SUPPORT_URL + ".",
        "pt": "Atualização: a correção já está na versão {fixed}. Atualize o jogo pela Google Play. Se ainda tiver problemas, fale conosco em " + SUPPORT_URL + ".",
        "he": "עדכון: התיקון זמין בגרסה {fixed}. עדכנו את המשחק דרך Google Play. אם עדיין יש בעיה, כתבו לנו ב־" + SUPPORT_URL + ".",
    },
    "incident": {
        "en": "Sorry about this! We know about the issue in version {version} on some devices and the team is working on a fix. We'll update this reply when it's out.",
        "de": "Das tut uns leid! Wir kennen das Problem in Version {version} auf einigen Geräten und arbeiten an einer Lösung. Wir aktualisieren diese Antwort, sobald sie verfügbar ist.",
        "pt": "Sentimos muito! Já sabemos do problema na versão {version} em alguns aparelhos e a equipe está trabalhando na correção. Vamos atualizar esta resposta quando sair.",
        "he": "מצטערים על זה! אנחנו מכירים את הבעיה בגרסה {version} בחלק מהמכשירים, והצוות עובד על תיקון. נעדכן את התשובה הזו כשהוא ייצא.",
    },
}
# drafts for people (never auto-posted; an agent edits and sends)
DRAFTS = {
    "SAFETY": "Thank you for telling us, and we're very sorry. Keeping kids safe is our top priority. Our safety team is reviewing this now. Please also use Report in the player's profile so we can act on the account. We'd like to follow up with you at " + SUPPORT_URL + ".",
    "LEGAL": "Thank you for raising this. We take it seriously and our team will look into it. Please contact us at " + SUPPORT_URL + " so we can follow up with you directly.",
    "payment": "Sorry about the purchase problem. Please open Settings > Support in the game so we can see your order and fix it. Never share card details in a review.",
    "progress": "We're sorry your progress is missing. Please open Settings > Support in the game. With your player ID our team can look into restoring it.",
    "default": "Thanks for the detailed feedback, and sorry for the trouble. Our team is looking into it. You can reach us anytime at " + SUPPORT_URL + ".",
}

# ---------------------------------------------------------------------------
# POLICY GUARD - every automatic reply must pass ALL checks
# ---------------------------------------------------------------------------
CHECKS = [
    ("too_long", lambda r: len(r) > MAX_LEN),
    ("html", lambda r: bool(re.search(r"<[^>]+>", r))),
    ("asks_for_rating", lambda r: bool(re.search(
        r"(rating|stars?|review|bewertung|sterne|avalia|estrelas|דירוג|כוכבים).{0,40}(change|update|raise|higher|änder|erhöh|mud|aument|לשנות|להעלות)|"
        r"(change|update|raise|änder|mude|atualize|שנו|תעלו).{0,40}(rating|stars?|review|bewertung|sterne|avalia|estrelas|דירוג|כוכבים)", r, re.I))),
    ("incentive", lambda r: bool(re.search(
        r"(free|gratis|grátis|חינם|בחינם|reward|belohnung|recompensa|(?<![א-ת])פרס(?![א-ת])|bonus|coins|gems|münzen|moedas|מטבעות|code|código|קוד|discount|rabatt|desconto|הנחה)", r, re.I))),
    ("foreign_link", lambda r: any(SUPPORT_URL not in u for u in re.findall(r"(?:https?://)?(?:[\w-]+\.)+(?:com|net|org|io|app|example|ly)\S*", r))),
    ("email_address", lambda r: bool(re.search(r"[\w.+-]+@[\w-]+\.\w+", r))),
    ("asks_personal_data", lambda r: bool(re.search(
        r"password|passwort|senha|סיסמ|card number|kartennummer|número do cartão|מספר כרטיס|phone number|telefon|telefone|טלפון|home address|adresse|endereço|כתובת מגורים", r, re.I))),
    ("promises_money", lambda r: bool(re.search(r"refund|erstatt|reembols|החזר", r, re.I))),
    ("blames_user", lambda r: bool(re.search(r"your fault|you did it wrong|deine schuld|culpa sua|אשמתך", r, re.I))),
]


def policy_check(reply):
    return [name for name, bad in CHECKS if bad(reply)]


# ---------------------------------------------------------------------------
def lane_for(r, active_incidents):
    flags = set((r["flags"] or "").split(",")) - {""}
    cat, score, rating = r["category"], r["category_score"], r["star_rating"]
    if r["is_noise"]:
        return ("NO_REPLY", "spam - reported to Google" if r["noise_reason"] == "spam" else f"noise: {r['noise_reason']}")
    if "child_safety" in flags:
        return ("SAFETY", "child safety report")
    if cat == "chat_safety":
        # the MODEL thinks it's about safety but the RULES found nothing: a person checks, Legal is NOT paged.
        # (real example from this data: Hebrew "my kids are dying for it" = "my kids love it")
        return ("HUMAN_CHECK", "possible safety topic - verify")
    if flags & {"legal", "privacy", "store_policy"}:
        names = {"legal": "legal threat", "privacy": "privacy / kids' data", "store_policy": "store-policy risk"}
        return ("LEGAL", " + ".join(names[f] for f in sorted(flags & set(names))))
    vip = []
    if "payment" in flags or cat == "payment_purchase":
        vip.append("payment")
    if cat == "progress_account_loss":
        vip.append("lost progress")
    if "high_value" in flags:
        vip.append("high-value player")
    if "rating_drop" in flags:
        vip.append(f"rating drop {r['first_star_rating']}->{rating}")
    if vip:
        return ("HUMAN_VIP", ", ".join(vip))
    if (r["app"], cat, r["version"]) in active_incidents:
        return ("INCIDENT", f"known issue: {cat} in v{r['version']}")
    if cat == "crash_stability":           # before the confidence check: a crash is never "unclear"
        return ("HUMAN_CHECK", "crash outside a known incident" + (" (5-star review!)" if rating >= 4 else ""))
    if cat == "other" or score is None or score < LOW_CONFIDENCE:
        if rating <= 2:
            return ("HUMAN_CHECK", "angry but unclear topic")
        return ("NO_REPLY", "unclear topic, not negative")
    if cat in T:
        return ("AUTO", f"template: {cat}")
    return ("HUMAN_CHECK", f"no approved template for {cat}")


TEAM = {"SAFETY": ("Trust & Safety + Legal", 1), "LEGAL": ("Legal", 24), "HUMAN_VIP": ("Player Support", 24),
        "HUMAN_CHECK": ("Community team", 48), "INCIDENT": ("auto", 0), "AUTO": ("auto", 0), "NO_REPLY": ("-", 0)}
BASE_PRIORITY = {"SAFETY": 100, "LEGAL": 80, "HUMAN_VIP": 60, "HUMAN_CHECK": 30}


def priority(lane, r, reason=""):
    if lane not in BASE_PRIORITY:
        return 0
    p = BASE_PRIORITY[lane] + (40 if "possible safety" in reason else 0)
    p += min(15, (r["thumbs_up"] or 0) // 2)                  # public exposure: others agreed with it
    p += 10 if "high_value" in (r["flags"] or "") else 0
    p += 5 if "rating_drop" in (r["flags"] or "") else 0
    return p


def draft_for(lane, reason, r):
    lang = r["reviewer_language"] if r["reviewer_language"] in ("en", "de", "pt", "he") else "en"
    if lane == "AUTO":
        return T[r["category"]][lang].format(app=r["app"])
    if lane == "INCIDENT":
        return T["incident"][lang].format(version=r["version"])
    if lane in ("SAFETY", "LEGAL"):
        return DRAFTS[lane]
    if lane == "HUMAN_VIP":
        return DRAFTS["payment" if "payment" in reason else "progress" if "progress" in reason else "default"]
    if lane == "HUMAN_CHECK":
        return DRAFTS["default"]
    return None


# ---------------------------------------------------------------------------
def run():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    rows = [dict(r) for r in con.execute("""
        SELECT r.review_id, a.app_name AS app, r.text, r.star_rating, r.first_star_rating, r.reviewer_language,
               r.app_version_name AS version, r.thumbs_up, substr(r.last_modified_at,1,10) AS day,
               ra.is_noise, ra.noise_reason, ra.category, ra.category_score, ra.flags
        FROM reviews r JOIN apps a USING(package_name) JOIN review_analysis ra USING(review_id)
        ORDER BY r.last_modified_at""")]

    # incidents from step 5: (app, category, version) that had a RELEASE alert, active from first_raised
    incidents = {}
    for key, first in con.execute("SELECT alert_key, first_raised FROM alerts WHERE type='RELEASE'"):
        _, app, cat, ver = key.split("|")
        incidents[(app, cat, ver)] = first

    out, notes, posted_per_app_day = [], [], Counter()
    for r in rows:
        active = {k for k, first in incidents.items() if r["day"] >= first}
        # reviews written before the alert existed are handled when the incident opens
        lane, reason = lane_for(r, set(incidents))
        if lane == "INCIDENT":
            open_day = incidents[(r["app"], r["category"], r["version"])]
            r["reply_day"] = max(r["day"], open_day)
        else:
            r["reply_day"] = r["day"]
        team, sla = TEAM[lane]
        draft = draft_for(lane, reason, r)
        violations, status = [], "none"
        if lane in ("AUTO", "INCIDENT"):
            violations = policy_check(draft)
            quota_key = (r["app"], r["reply_day"])
            if violations:
                status, lane = "blocked_by_policy -> person", "HUMAN_CHECK"
            elif posted_per_app_day[quota_key] >= DAILY_QUOTA:
                status = "deferred_quota"
            else:
                posted_per_app_day[quota_key] += 1
                status = "posted"
                if rng.random() < QA_SAMPLE:
                    status = "posted + QA sample"
        elif lane in BASE_PRIORITY:
            status = "queued_for_person"
        if lane in ("SAFETY", "LEGAL"):
            for t in team.split(" + "):
                notes.append((t, r["review_id"], r["day"], f"{lane}: {r['app']} - {r['text'][:120]}"))
        out.append((r["review_id"], lane, reason, priority(lane, r, reason), team, sla, draft, status, ",".join(violations)))

    con.execute("DROP TABLE IF EXISTS triage")
    con.execute("""CREATE TABLE triage (review_id TEXT PRIMARY KEY, lane TEXT, reason TEXT, priority INTEGER, team TEXT,
                   sla_hours INTEGER, draft TEXT, reply_status TEXT, violations TEXT)""")
    con.executemany("INSERT INTO triage VALUES (?,?,?,?,?,?,?,?,?)", out)
    con.execute("DROP TABLE IF EXISTS notifications")
    con.execute("CREATE TABLE notifications (team TEXT, review_id TEXT, sent_on TEXT, message TEXT)")
    con.executemany("INSERT INTO notifications VALUES (?,?,?,?)", notes)
    # INCIDENT FOLLOW-UP: when a newer version ships and the problem's share drops back, edit every incident reply
    followups = []
    for (app, cat, ver), opened in incidents.items():
        later = sorted({r["version"] for r in rows if r["app"] == app and r["version"] and
                        [int(x) for x in r["version"].split(".")] > [int(x) for x in ver.split(".")]})
        for nv in later:
            nv_rows = [r for r in rows if r["app"] == app and r["version"] == nv and not r["is_noise"]]
            if len(nv_rows) >= 50 and sum(r["category"] == cat for r in nv_rows) / len(nv_rows) < 0.05:
                fixed_on = min(r["day"] for r in nv_rows)
                followups.append((app, cat, ver, nv, fixed_on))
                break
    by_id = {r["review_id"]: r for r in rows}
    edited = 0
    for app, cat, ver, nv, fixed_on in followups:
        for i, o in enumerate(out):
            r = by_id[o[0]]
            if o[1] == "INCIDENT" and r["app"] == app and r["version"] == ver and o[7].startswith("posted"):
                lang = r["reviewer_language"] if r["reviewer_language"] in T["incident_fixed"] else "en"
                new = T["incident_fixed"][lang].format(fixed=nv)
                if not policy_check(new):
                    out[i] = o[:6] + (new,) + (o[7] + f", edited {fixed_on} (fixed in {nv})", o[8])
                    edited += 1
    print(f"Incident follow-ups: {[(a, f'v{v} fixed in v{nv} on {d}') for a, _, v, nv, d in followups]}, "
          f"{edited} replies edited\n")
    con.execute("DELETE FROM triage")
    con.executemany("INSERT INTO triage VALUES (?,?,?,?,?,?,?,?,?)", out)

    # simulate posting
    posted = {o[0]: o[6] for o in out if o[7].startswith("posted")}
    day_of = {r["review_id"]: r["reply_day"] for r in rows}
    con.executemany("UPDATE reviews SET developer_reply=?, replied_at=? WHERE review_id=?",
                    [(txt, day_of[rid], rid) for rid, txt in posted.items()])
    con.commit()
    report(con, rows, out, posted_per_app_day, notes)
    con.close()


def report(con, rows, out, quota, notes):
    n = len(out)
    lanes = Counter(o[1] for o in out)
    print(f"=== {n:,} reviews over 60 days ===\n")
    order = ["SAFETY", "LEGAL", "HUMAN_VIP", "HUMAN_CHECK", "INCIDENT", "AUTO", "NO_REPLY"]
    for l in order:
        print(f"  {l:12s} {lanes[l]:6,}  ({lanes[l] / n:5.1%})   -> {TEAM[l][0]}")
    human = sum(lanes[l] for l in BASE_PRIORITY)
    auto = lanes["AUTO"] + lanes["INCIDENT"]
    print(f"\n  handled without a person: {(auto + lanes['NO_REPLY']) / n:.0%}   "
          f"people needed for: {human:,} reviews (~{human / 60:.0f} a day instead of ~{n / 60:.0f})")
    st = Counter(o[7] for o in out)
    print(f"  automatic replies posted: {st['posted'] + st['posted + QA sample']:,}  "
          f"(QA sample for a person: {st['posted + QA sample']})  blocked by policy guard: "
          f"{sum(1 for o in out if o[7].startswith('blocked'))}")
    print(f"  busiest app-day for automatic replies: {max(quota.values())} of {DAILY_QUOTA} allowed")
    print(f"  notifications sent: {dict(Counter(t for t, *_ in notes))}")

    print("\nTop of the human queue:")
    texts = {r["review_id"]: r for r in rows}
    for o in sorted([o for o in out if o[1] in BASE_PRIORITY], key=lambda o: -o[3])[:6]:
        r = texts[o[0]]
        print(f"  [{o[3]:3d}] {o[1]:11s} {r['app']:12s} {r['star_rating']}★  {o[2]:28s} \"{r['text'][:70]}\"")

    print("\nPolicy guard self-test (replies that must be blocked):")
    bad = ["Sorry! Please update your rating to 5 stars once it's fixed.",
           "Thanks! Here's a code for 100 free coins: GIFT100",
           "Contact us at help@example.com",
           "Download our other game at othergame.example!",
           "Send us your password and we'll check your account.",
           "We will refund you right away.",
           "Bitte ändere deine Bewertung auf 5 Sterne.",
           "תעלו את הדירוג ונשלח לכם מטבעות בחינם",
           "x" * 400]
    for b in bad:
        print(f"  {'BLOCKED' if policy_check(b) else 'PASSED!!'}  {policy_check(b)}  \"{b[:55]}\"")
    clean = sum(not policy_check(t[l].format(app="PeleGame", version="2.4.0", fixed="2.4.1")) for t in T.values() for l in t)
    print(f"  all {sum(len(t) for t in T.values())} approved templates pass: {clean == sum(len(t) for t in T.values())}")


if __name__ == "__main__":
    run()


In [ ]:
!python triage.py

## Step 6b: urgent escalations

In [ ]:
%%writefile urgent.py
"""
Step 6b - Urgent escalations (hourly, rules only)
=================================================

Some reviews can't wait for the morning report: a child in danger, a legal
threat, or a report that the app breaks a Google Play policy (gambling or
adult ads in a kids game, ads that install apps, kids' data). These can get
an app suspended from the store, so the product manager of that app and Legal
must hear about them within the hour.

How it runs:
  * Every hour (not daily): pull the newest reviews and run ONLY the keyword
    rules from classify.py. No model, takes seconds, can't be "unsure".
  * Each hit becomes or joins an alert for (app, type). Reports on the same
    problem within 72 hours join the open alert instead of paging people again.
  * The alert pages the owners for that app and type, with a deadline (SLA).
    3+ reports in 24h escalates to the Head of Product.
  * These reviews never get an automatic reply. A person drafts the answer and
    Legal approves it before anything is posted publicly.

Output (in reviews.db):
  urgent_alerts         one row per alert: type, app, first/last report, detected_at,
                        paged roles, SLA, policy at risk, actions, message
  urgent_notifications  one row per person paged

Run after triage.py:  python urgent.py
"""

import json
import sqlite3
from datetime import datetime, timedelta

DB = "reviews.db"
CHECK_EVERY = timedelta(hours=1)
MERGE_WINDOW = timedelta(hours=72)
DASHBOARD = "Review Morning Desk"

# who owns what (fake contacts for the POC - in production this lives in a config sheet)
OWNERS = {
    "pm": {"Merge Meadow": "pm.mergemeadow@example.com", "Smash Runner": "pm.smashrunner@example.com",
           "Tile Twist": "pm.tiletwist@example.com", "PeleGame": "pm.pelegame@example.com"},
    "legal": "legal-oncall@example.com",
    "trust_safety": "trust-safety@example.com",
    "dpo": "privacy@example.com",
    "ad_ops": "ad-monetization@example.com",
    "support_lead": "support-lead@example.com",
    "head_of_product": "head-of-product@example.com",
}

TYPES = {   # checked in this order; a review belongs to the first type it matches
    "CHILD_SAFETY": dict(flags={"child_safety"}, sla_h=1, label="Child safety",
                         roles=["pm", "legal", "trust_safety"],
                         policy="Google Play child safety standards and Families policy",
                         actions=["Trust & Safety reviews the reported account and chat logs now",
                                  "Preserve evidence; Legal decides on reporting to authorities",
                                  "Personal reply drafted by Trust & Safety, approved by Legal"]),
    "STORE_POLICY": dict(flags={"store_policy"}, sla_h=4, label="Store policy risk",
                         roles=["pm", "legal", "ad_ops"],
                         policy="Google Play Ads policy and Families policy (ads in apps used by children)",
                         actions=["Ad ops checks which ad network served it and blocks that category/network",
                                  "PM decides whether to ship a hotfix or pause the network remotely",
                                  "No public reply until Legal approves the wording"]),
    "LEGAL": dict(flags={"legal"}, sla_h=4, label="Legal threat",
                  roles=["legal", "pm", "support_lead"],
                  policy="Consumer protection and payments (refund and billing obligations)",
                  actions=["Support lead checks the player's orders today",
                           "Legal approves any reply that mentions money or responsibility"]),
    "PRIVACY": dict(flags={"privacy"}, sla_h=24, label="Privacy / kids' data",
                    roles=["dpo", "legal", "pm"],
                    policy="Google Play User Data policy and the Data safety section",
                    actions=["Privacy team checks which permissions this version requests and why",
                             "Confirm the Data safety form matches what the app really collects"]),
}


def detected_at(ts):
    """The hourly job sees a review at the next full hour, and pages 5 minutes later."""
    nxt = (ts + CHECK_EVERY).replace(minute=0, second=0, microsecond=0)
    return nxt + timedelta(minutes=5)


def run():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    rows = [dict(r) for r in con.execute("""
        SELECT r.review_id, a.app_name AS app, r.text, r.star_rating, r.reviewer_language AS lang,
               r.app_version_name AS version, r.thumbs_up, r.last_modified_at AS ts, ra.flags
        FROM reviews r JOIN apps a USING(package_name) JOIN review_analysis ra USING(review_id)
        WHERE ra.flags <> '' ORDER BY r.last_modified_at""")]

    alerts, open_by_key = [], {}
    for r in rows:
        flags = set(r["flags"].split(","))
        typ = next((t for t, cfg in TYPES.items() if cfg["flags"] & flags), None)
        if not typ:
            continue
        ts = datetime.fromisoformat(r["ts"])
        det = detected_at(ts)
        key = (r["app"], typ)
        a = open_by_key.get(key)
        if a and ts - a["last_report"] <= MERGE_WINDOW:
            a["reviews"].append(r)
            a["last_report"] = ts
            recent = [x for x in a["reviews"] if ts - datetime.fromisoformat(x["ts"]) <= timedelta(hours=24)]
            if len(recent) >= 3 and not a["escalated_at"]:
                a["escalated_at"] = det
            continue
        cfg = TYPES[typ]
        a = dict(id=f"U{len(alerts) + 1:03d}", type=typ, app=r["app"], first_report=ts, last_report=ts,
                 detected_at=det, escalated_at=None, reviews=[r], cfg=cfg)
        alerts.append(a)
        open_by_key[key] = a

    con.execute("DROP TABLE IF EXISTS urgent_alerts")
    con.execute("""CREATE TABLE urgent_alerts (alert_id TEXT PRIMARY KEY, type TEXT, label TEXT, app TEXT,
                   first_report TEXT, last_report TEXT, detected_at TEXT, minutes_to_page INTEGER, escalated_at TEXT,
                   reports INTEGER, sla_hours INTEGER, due_by TEXT, paged TEXT, policy TEXT, actions TEXT,
                   review_ids TEXT, message TEXT)""")
    con.execute("DROP TABLE IF EXISTS urgent_notifications")
    con.execute("CREATE TABLE urgent_notifications (alert_id TEXT, role TEXT, address TEXT, sent_at TEXT, reason TEXT)")

    for a in alerts:
        cfg, first = a["cfg"], a["reviews"][0]
        paged = []
        for role in cfg["roles"]:
            addr = OWNERS[role][a["app"]] if role == "pm" else OWNERS[role]
            paged.append((role, addr, a["detected_at"].isoformat(), "new alert"))
        if a["escalated_at"]:
            paged.append(("head_of_product", OWNERS["head_of_product"], a["escalated_at"].isoformat(), "3+ reports in 24h"))
        due = a["detected_at"] + timedelta(hours=cfg["sla_h"])
        mins = int((a["detected_at"] - a["first_report"]).total_seconds() // 60)
        msg = (f"URGENT · {cfg['label']} · {a['app']}\n"
               f"{len(a['reviews'])} report(s) since {a['first_report']:%d %b %H:%M} UTC. Respond by {due:%d %b %H:%M} UTC ({cfg['sla_h']}h).\n"
               f"At risk: {cfg['policy']}.\n"
               f"Latest review ({first['star_rating']}★, {first['lang']}, v{first['version']}): \"{a['reviews'][-1]['text'][:200]}\"\n"
               f"Next steps: " + "; ".join(cfg["actions"]) + ".\n"
               f"Automatic replies are blocked for these reviews. Open {DASHBOARD} for all reports.")
        con.execute("INSERT INTO urgent_alerts VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    (a["id"], a["type"], cfg["label"], a["app"], a["first_report"].isoformat(), a["last_report"].isoformat(),
                     a["detected_at"].isoformat(), mins, a["escalated_at"].isoformat() if a["escalated_at"] else None,
                     len(a["reviews"]), cfg["sla_h"], due.isoformat(), json.dumps([p[:2] for p in paged]), cfg["policy"],
                     json.dumps(cfg["actions"]), json.dumps([x["review_id"] for x in a["reviews"]]), msg))
        con.executemany("INSERT INTO urgent_notifications VALUES (?,?,?,?,?)", [(a["id"], *p) for p in paged])
    con.commit()

    print(f"{len(alerts)} urgent alerts from {sum(len(a['reviews']) for a in alerts)} reports\n")
    for a in alerts:
        mins = int((a["detected_at"] - a["first_report"]).total_seconds() // 60)
        print(f"  {a['id']} {a['cfg']['label']:20s} {a['app']:13s} reports={len(a['reviews']):<3} "
              f"first {a['first_report']:%d %b %H:%M} -> paged {a['detected_at']:%H:%M} ({mins} min)"
              f"{'  ESCALATED to Head of Product' if a['escalated_at'] else ''}")
        print(f"       paged: {', '.join(a['cfg']['roles'])}   SLA {a['cfg']['sla_h']}h")
    worst = max(int((a["detected_at"] - a["first_report"]).total_seconds() // 60) for a in alerts)
    print(f"\nSlowest time from review to page: {worst} minutes")
    con.close()


if __name__ == "__main__":
    run()


In [ ]:
!python urgent.py

## Step 7: dashboard data

In [ ]:
%%writefile export_dashboard.py
"""
Step 7 - Export everything the dashboard needs from reviews.db into one JSON file.
In production this runs right after triage.py every morning, and the dashboard
reads the fresh file. Run:  python export_dashboard.py  -> dashboard_data.json
"""
import json
import random
import sqlite3
from collections import Counter, defaultdict

con = sqlite3.connect("reviews.db")
con.row_factory = sqlite3.Row
q = lambda sql, *a: [dict(r) for r in con.execute(sql, a)]

rows = q("""SELECT r.review_id, a.app_name AS app, substr(r.last_modified_at,1,10) AS day, r.star_rating AS stars,
                   r.first_star_rating AS first_stars, r.reviewer_language AS lang, r.app_version_name AS version,
                   r.device_name AS device, r.text, r.developer_reply AS reply, ra.category, ra.is_noise, ra.noise_reason,
                   t.lane, t.reason, t.priority, t.team, t.sla_hours, t.draft, t.reply_status
            FROM reviews r JOIN apps a USING(package_name) JOIN review_analysis ra USING(review_id)
            JOIN triage t USING(review_id) ORDER BY r.last_modified_at""")
apps = [r["app_name"] for r in q("SELECT app_name FROM apps")]
days = sorted({r["day"] for r in rows})

# per day per app: totals, lanes, categories (non-noise)
daily = defaultdict(lambda: {"total": 0, "lanes": Counter(), "cats": Counter(), "langs": Counter()})
for r in rows:
    d = daily[(r["day"], r["app"])]
    d["total"] += 1
    d["lanes"][r["lane"]] += 1
    d["langs"][r["lang"]] += 1
    if not r["is_noise"] and r["category"]:
        d["cats"][r["category"]] += 1
daily_out = {f"{k[0]}|{k[1]}": {"t": v["total"], "l": dict(v["lanes"]), "c": dict(v["cats"])} for k, v in daily.items()}

# collector log (what the 06:00 job saw)
runs = defaultdict(lambda: {"fetched": 0, "new": 0, "edited": 0, "repeats": 0})
for r in q("SELECT substr(run_at,1,10) AS d, fetched, new_reviews, edited, repeats FROM ingestion_runs"):
    x = runs[r["d"]]
    x["fetched"] += r["fetched"]; x["new"] += r["new_reviews"]; x["edited"] += r["edited"]; x["repeats"] += r["repeats"]

# releases: first day each version shows up
releases = defaultdict(dict)
for r in rows:
    if r["version"] and r["version"] not in releases[r["app"]]:
        releases[r["app"]][r["version"]] = r["day"]

# issues + their signals + themes
alerts = q("SELECT alert_key, type, app, category, first_raised, evidence FROM alerts")
themes = {t["theme_id"]: t for t in q("SELECT theme_id, genre, category, keywords, quote, born_on, size, languages FROM themes")}
issues = []
for i in q("SELECT * FROM issues"):
    sigs = [dict(type=a["type"], day=a["first_raised"], ev=json.loads(a["evidence"])) for a in alerts
            if a["app"] == i["app"] and a["category"] == i["category"]]
    for s in sigs:
        s["ev"].pop("review_ids", None)
    th = [dict(id=t, **{k: themes[t][k] for k in ("keywords", "quote", "born_on", "size")},
               langs=json.loads(themes[t]["languages"])) for t in json.loads(i["themes"]) if t in themes]
    ev = json.loads(i["evidence"])
    ev.pop("review_ids", None)
    issues.append(dict(app=i["app"], category=i["category"], lead=i["lead_signal"], first=i["first_raised"],
                       priority=i["priority"], signals=sorted(sigs, key=lambda s: s["day"]), themes=th, ev=ev))

# human queue (all), auto-reply samples (a few per day)
queue = [dict(id=r["review_id"], day=r["day"], app=r["app"], stars=r["stars"], first=r["first_stars"], lang=r["lang"],
              version=r["version"], text=r["text"], lane=r["lane"], reason=r["reason"], p=r["priority"], team=r["team"],
              sla=r["sla_hours"], draft=r["draft"]) for r in rows if r["lane"] in ("SAFETY", "LEGAL", "HUMAN_VIP", "HUMAN_CHECK")]
rng = random.Random(3)
by_day = defaultdict(list)
for r in rows:
    if r["reply_status"] and r["reply_status"].startswith("posted"):
        by_day[r["day"]].append(r)
samples = []
for d, lst in by_day.items():
    inc = [r for r in lst if r["lane"] == "INCIDENT"]
    pick = rng.sample(inc, min(2, len(inc))) + rng.sample([r for r in lst if r["lane"] == "AUTO"], min(4, len(lst)))
    for r in pick:
        samples.append(dict(day=d, app=r["app"], stars=r["stars"], lang=r["lang"], cat=r["category"], text=r["text"],
                            reply=r["reply"], lane=r["lane"], status=r["reply_status"]))

# events the run log should mention
import re
from triage import T
events = defaultdict(list)
edits = Counter()
for r in rows:
    m = re.search(r"edited (\d{4}-\d\d-\d\d) \(fixed in ([\d.]+)\)", r["reply_status"] or "")
    if m:
        edits[(m.group(1), r["app"], r["version"], m.group(2))] += 1
for (d, app, v, nv), n in edits.items():
    events[d].append(f"{app} v{nv} fixed the v{v} incident: {n} earlier replies edited to say the fix is out")
for s_ in samples:
    m = re.search(r"edited (\d{4}-\d\d-\d\d)", s_["status"] or "")
    if m and s_["lane"] == "INCIDENT":
        lang = s_["lang"] if s_["lang"] in T["incident"] else "en"
        ver = next((r["version"] for r in rows if r["text"] == s_["text"]), "")
        s_["reply_before"] = T["incident"][lang].format(version=ver)
        s_["edited_on"] = m.group(1)

# urgent alerts (hourly channel)
texts = {r["review_id"]: r for r in rows}
urgent = []
for u in q("SELECT * FROM urgent_alerts ORDER BY detected_at"):
    revs = [dict(ts=t["ts"], text=texts[i]["text"], stars=texts[i]["stars"], lang=texts[i]["lang"], version=texts[i]["version"])
            for i in json.loads(u["review_ids"]) for t in [dict(ts=con.execute("SELECT last_modified_at FROM reviews WHERE review_id=?", (i,)).fetchone()[0])]]
    urgent.append(dict(id=u["alert_id"], type=u["type"], label=u["label"], app=u["app"], first=u["first_report"],
                       detected=u["detected_at"], mins=u["minutes_to_page"], escalated=u["escalated_at"], sla=u["sla_hours"],
                       due=u["due_by"], paged=json.loads(u["paged"]), policy=u["policy"], actions=json.loads(u["actions"]),
                       message=u["message"], reviews=revs))

data = dict(urgent=urgent, events=events, apps=apps, days=days, daily=daily_out, runs=runs, releases=releases, issues=issues, queue=queue,
            samples=samples, totals=dict(reviews=len(rows), languages=len({r["lang"] for r in rows})))
json.dump(data, open("dashboard_data.json", "w", encoding="utf-8"), ensure_ascii=False, separators=(",", ":"))
print(f"days {len(days)}, issues {len(issues)}, queue {len(queue)}, samples {len(samples)}")
import os; print(f"{os.path.getsize('dashboard_data.json') / 1024:.0f} KB")


In [ ]:
!python export_dashboard.py

## Check against the planted ground truth

In [ ]:
%%writefile evaluate.py
"""
Check step 3 against the planted ground truth (the pipeline never reads this file).
Run after classify.py:   python evaluate.py
"""
import csv
import json
import sqlite3
from collections import Counter, defaultdict

EXPECTED_CATEGORY = {
    "version_crash": "crash_stability", "rating_mismatch": "crash_stability",
    "progress_lost": "progress_account_loss", "iap_not_received": "payment_purchase",
    "ads_chronic": "ads", "feature_avatars": "feature_request", "child_safety": "chat_safety",
}
EXPECTED_FLAG = {"child_safety": "child_safety", "iap_not_received": "payment", "high_value": "high_value"}

truth = {r["review_id"]: r["true_scenario"] for r in csv.DictReader(open("data/ground_truth.csv", encoding="utf-8"))}
con = sqlite3.connect("reviews.db")
pred = {rid: (noise, reason, cat, (flags or "").split(","), mm) for rid, noise, reason, cat, flags, mm in
        con.execute("SELECT review_id, is_noise, noise_reason, category, flags, rating_mismatch FROM review_analysis")}

# noise
is_noise_true = {rid for rid, s in truth.items() if s.startswith("noise")}
is_noise_pred = {rid for rid, p in pred.items() if p[0]}
tp = len(is_noise_true & is_noise_pred)
print(f"NOISE   caught {tp}/{len(is_noise_true)} ({tp / len(is_noise_true):.0%})   "
      f"false alarms {len(is_noise_pred - is_noise_true)} real reviews wrongly dropped")
missed = Counter(truth[r] for r in is_noise_true - is_noise_pred)
if missed:
    print("        missed by type:", dict(missed))

# categories
print("\nCATEGORY  (planted scenario -> share that got the expected category)")
by = defaultdict(Counter)
for rid, s in truth.items():
    if s in EXPECTED_CATEGORY:
        by[s][pred[rid][2]] += 1
for s, cnt in by.items():
    tot = sum(cnt.values())
    ok = cnt[EXPECTED_CATEGORY[s]]
    others = ", ".join(f"{c} {n}" for c, n in cnt.most_common(4) if c != EXPECTED_CATEGORY[s])
    print(f"  {s:18s} {ok}/{tot} ({ok / tot:.0%})   {('else: ' + others) if others else ''}")

# error analysis: show a few misses so we can see WHY
texts = dict(con.execute("SELECT review_id, text FROM reviews"))
for s in ("rating_mismatch", "progress_lost", "feature_avatars"):
    miss = [rid for rid, t in truth.items() if t == s and pred[rid][2] != EXPECTED_CATEGORY[s]][:3]
    for rid in miss:
        print(f"    miss [{s} -> {pred[rid][2]}] {texts[rid][:90]}")

# flags (recall is what matters here)
print("\nRISK FLAGS  (must not miss)")
for s, flag in EXPECTED_FLAG.items():
    ids = [rid for rid, t in truth.items() if t == s]
    hit = sum(flag in pred[rid][3] for rid in ids)
    print(f"  {s:18s} -> {flag:13s} {hit}/{len(ids)} ({hit / len(ids):.0%})")
cs_flagged = [rid for rid, p in pred.items() if "child_safety" in p[3]]
print(f"  child_safety flags raised in total: {len(cs_flagged)} "
      f"(real: {sum(truth[r] == 'child_safety' for r in cs_flagged)}, extra for a human to glance at: "
      f"{sum(truth[r] != 'child_safety' for r in cs_flagged)})")
mm = [rid for rid, t in truth.items() if t == "rating_mismatch"]
print(f"\nRATING MISMATCH  caught {sum(pred[r][4] for r in mm)}/{len(mm)}")

# ---------------------------------------------------------------------------
# THEMES (step 4): were planted problems discovered as themes, and how fast?
try:
    th = {tid: (born, size, kw) for tid, born, size, kw in con.execute("SELECT theme_id, born_on, size, keywords FROM themes")}
    rt = dict(con.execute("SELECT review_id, theme_id FROM review_themes"))
    first_day = {}
    for rid, day in con.execute("SELECT review_id, substr(last_modified_at,1,10) FROM reviews"):
        s_ = truth[rid]
        first_day[s_] = min(first_day.get(s_, day), day)
    print("\nTHEMES  (planted problem -> which themes hold it, and when the first theme was born)")
    for s_ in ("version_crash", "progress_lost", "iap_not_received", "feature_avatars", "ads_chronic"):
        ids = [rid for rid, t in truth.items() if t == s_ and rid in rt]
        tids = Counter(rt[rid] for rid in ids if rt[rid])
        covered = sum(tids.values())
        top = tids.most_common(3)
        born = min((th[t][0] for t, _ in top), default="-")
        purity = ", ".join(f"{t}: {n}/{th[t][1]}" for t, n in top)
        print(f"  {s_:17s} started {first_day[s_]}  first theme {born}  in themes {covered}/{len(ids)}  top: {purity}")
except sqlite3.OperationalError:
    pass

# ---------------------------------------------------------------------------
# ALERTS (step 5): did each planted problem become an issue, and how many days after it started?
PLANTED = [("version_crash", "Merge Meadow", "crash_stability"), ("progress_lost", "Tile Twist", "progress_account_loss"),
           ("iap_not_received", "PeleGame", "payment_purchase"), ("child_safety", "PeleGame", "child_safety"),
           ("feature_avatars", "PeleGame", "feature_request"), ("ads_chronic", "Smash Runner", "ads")]
try:
    issues = {(a, c): (lead, first, sig) for a, c, lead, first, sig in
              con.execute("SELECT app, category, lead_signal, first_raised, signals FROM issues")}
    from datetime import date as _d
    print("\nALERTS  (planted problem -> issue raised?  days from first review to alert)")
    for s_, app, cat in PLANTED:
        start = first_day.get(s_)
        if (app, cat) in issues:
            lead, first, sig = issues[(app, cat)]
            delay = (_d.fromisoformat(first) - _d.fromisoformat(start)).days
            print(f"  {s_:17s} YES  {lead:9s} started {start} -> raised {first}  ({delay} days)  signals: {', '.join(json.loads(sig))}")
        else:
            print(f"  {s_:17s} NO   (started {start})")
    false_alarms = [k for k in issues if k not in {(a, c) for _, a, c in PLANTED}]
    print(f"  other issues raised (check if real or false alarms): {false_alarms}")
except sqlite3.OperationalError:
    pass


In [ ]:
!python evaluate.py

## Download

In [ ]:
from google.colab import files
files.download('reviews.db'); files.download('dashboard_data.json')